# Delivery Timing and Customer Dissatisfaction in E-Commerce
### An Asymmetry Analysis Across Product Categories — selected analysis code

Master Thesis, Data Science and Marketing Analytics, Erasmus School of Economics, Erasmus University Rotterdam
Sinem Kırcalı · Supervisor: Hans Ligtenberg · Second assessor: Nuno Camacho

This notebook is a **partial release** of the analysis code behind the thesis. Parts 0–2 (setup, data and sample
construction, and the machine-learning classification for H1) are published here with their saved outputs. The code
for Parts 3–5 is not published; the README explains why. The result tables and figures of all phases are in `output/`.
The parts follow the structure of the thesis:

| Notebook part | Thesis section | Main tables / figures | Code |
|---|---|---|---|
| **Part 0** Setup | – | – | Published |
| **Part 1** Data and sample construction | §3, §4.1 | Table 1, Table 2, Appendix J.1, J.7, K | Published |
| **Part 2** Phase 1 — Machine-learning classification (H1) | §4.2, §5.1 | Tables 3–6, Figure 1, Appendix A, C, G | Published |
| **Part 3** Phase 2 — Asymmetry test (H2a) | §4.3, §5.2.1–5.2.3 | Tables 7–8, Appendix B, D, H.1–H.3, H.5, H.6, J.2 | Withheld |
| **Part 4** Product-category moderation (H2b) | §4.4, §5.2.4 | Table 9, Appendix H.4, J.3 | Withheld |
| **Part 5** Phase 3 — Text analysis (H3) | §4.5, §5.3 | Tables 10–11, Figure 2, Appendix E, F, I, J.4–J.6 | Withheld |

The notebook is published for reading. It is not a replication package for the full thesis.


---
## Part 0 — Setup

Imports, paths and a check that all eight Olist source tables are present (§3.1).
Uncomment the `pip install` line once if a library is missing.

In [1]:
# Run once if a library is missing:
# !pip install --quiet numpy pandas matplotlib scikit-learn xgboost shap


In [2]:
# ============================================================
# IMPORTS — libraries used in the published parts
# ============================================================
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    f1_score,
    recall_score,
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# XGBoost
import xgboost as xgb

# SHAP
import shap


In [3]:
# ============================================================
# CONFIGURATION — paths and global constants
# ============================================================

# Works both in a .py script and in Jupyter
try:
    PROJECT_ROOT = os.path.dirname(
        os.path.abspath(__file__)
    )
except NameError:
    PROJECT_ROOT = os.getcwd()

# Your CSV files are directly inside the project folder
DATA_DIR = PROJECT_ROOT

OUT_DIR = os.path.join(
    PROJECT_ROOT,
    "output"
)

RANDOM_STATE = 42

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR:     {DATA_DIR}")
print(f"OUT_DIR:      {OUT_DIR}")

PROJECT_ROOT: .
DATA_DIR:     .
OUT_DIR:      output


In [4]:
required_files = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_customers_dataset.csv",
    "product_category_name_translation.csv",
]

print("\nFILE CHECK")

all_found = True

for filename in required_files:
    path = os.path.join(DATA_DIR, filename)
    exists = os.path.exists(path)

    print(
        f"{filename:45s} "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    all_found = all_found and exists

print(
    f"\nAll required files found: {all_found}"
)


FILE CHECK
olist_orders_dataset.csv                      FOUND
olist_order_items_dataset.csv                 FOUND
olist_order_payments_dataset.csv              FOUND
olist_order_reviews_dataset.csv               FOUND
olist_products_dataset.csv                    FOUND
olist_sellers_dataset.csv                     FOUND
olist_customers_dataset.csv                   FOUND
product_category_name_translation.csv         FOUND

All required files found: True


---
## Part 1 — Data and sample construction (Thesis §3 and §4.1)

### 1.1 Load and merge the eight Olist tables (§3.1–3.2)

The tables are merged at the **order level**, because the review score and the promised delivery date
are recorded per order:
- **order_items**: price and freight are summed over all items; the item with `order_item_id = 1`
  supplies the primary product, seller and product category.
- **order_payments**: total payment value, maximum number of installments, and the dominant payment
  type (the payment row with the highest value).
- **order_reviews**: for the 547 orders with more than one review record, the most recent review
  (by `review_creation_date`) is kept, so star rating and text always come from the same record.
- The two categories missing from the translation table (`pc_gamer`,
  `portateis_cozinha_e_preparadores_de_alimentos`; 19 orders) receive manual English names.

In [5]:
# ============================================================
# DATA PREPARATION
# ============================================================
# Section 1 of 5: loads raw Olist tables, merges them at the order
# level, applies completeness filters, and derives all features
# needed by H1–H3.  Two CSV artefacts are produced:
#   orders_merged.csv   — merged raw dataset
#   orders_features.csv — feature-engineered dataset
# (both written to the project folder, next to the raw tables)
# ============================================================

print("\n" + "=" * 60)
print("DATA PREPARATION — Step 1: Load & Merge")
print("=" * 60)

# -------------------------------------------------------------------
# 1. Read raw tables
# -------------------------------------------------------------------
orders              = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv")
order_items         = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
order_payments      = pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
order_reviews       = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")
products            = pd.read_csv(f"{DATA_DIR}/olist_products_dataset.csv")
sellers             = pd.read_csv(f"{DATA_DIR}/olist_sellers_dataset.csv")
customers           = pd.read_csv(f"{DATA_DIR}/olist_customers_dataset.csv")
category_translation = pd.read_csv(f"{DATA_DIR}/product_category_name_translation.csv")

# Two categories are absent from the translation table — add them manually
missing_translations = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "product_category_name_english": ["pc_gamer", "kitchen_portables_and_food_preparers"],
})
category_translation = pd.concat([category_translation, missing_translations], ignore_index=True)

print("Raw table shapes:")
for name, df_tmp in [
    ("orders", orders),
    ("order_items", order_items),
    ("order_payments", order_payments),
    ("order_reviews", order_reviews),
    ("products", products),
    ("sellers", sellers),
    ("customers", customers),
]:
    print(f"  {name:15s}: {df_tmp.shape}")

# -------------------------------------------------------------------
# 2. order_items: aggregate to order level
#    Most orders contain a single item; for multi-item orders we sum
#    price/freight and use the first item as the "main" product.
# -------------------------------------------------------------------
order_items_agg = (
    order_items.groupby("order_id")
    .agg(
        n_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
    )
    .reset_index()
)

# Keep product_id / seller_id of the primary item (order_item_id == 1)
main_item = order_items[order_items["order_item_id"] == 1][
    ["order_id", "product_id", "seller_id"]
]
order_items_merged = order_items_agg.merge(main_item, on="order_id", how="left")


DATA PREPARATION — Step 1: Load & Merge


Raw table shapes:
  orders         : (99441, 8)
  order_items    : (112650, 7)
  order_payments : (103886, 5)
  order_reviews  : (99224, 7)
  products       : (32951, 9)
  sellers        : (3095, 4)
  customers      : (99441, 5)


In [6]:
# -------------------------------------------------------------------
# 3. order_payments: aggregate to order level
#    An order may have multiple payment rows (e.g. instalment + voucher);
#    take the total value and the dominant payment type.
# -------------------------------------------------------------------
payments_total = (
    order_payments.groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        installment_count=("payment_installments", "max"),
    )
    .reset_index()
)

# Dominant type = row with the highest payment_value
dominant_payment = (
    order_payments.sort_values("payment_value", ascending=False)
    .groupby("order_id")
    .first()
    .reset_index()[["order_id", "payment_type"]]
)
payments_merged = payments_total.merge(dominant_payment, on="order_id", how="left")

# -------------------------------------------------------------------
# 4. order_reviews: keep only the most recent review per order
#    (theoretically multiple reviews are possible)
# -------------------------------------------------------------------
reviews_dedup = (
    order_reviews
    .sort_values("review_creation_date")
    .drop_duplicates(subset="order_id", keep="last")
)
# How many orders have multiple review records?
review_counts = (
    order_reviews.groupby("order_id")
    .size()
    .sort_values(ascending=False)
)

print("Orders with >1 review:", (review_counts > 1).sum())
print("Share of reviewed orders:", (review_counts > 1).mean())
print("Maximum reviews for one order:", review_counts.max())

print("\nDistribution:")
print(review_counts.value_counts().sort_index())

Orders with >1 review: 547
Share of reviewed orders: 0.00554356308209946
Maximum reviews for one order: 3

Distribution:
1    98126
2      543
3        4
Name: count, dtype: int64


In [7]:
# -------------------------------------------------------------------
# 5. Attach English category name to products
# -------------------------------------------------------------------
products_with_category = products.merge(
    category_translation, on="product_category_name", how="left"
)


# -------------------------------------------------------------------
# 6. Merge everything around the orders table
# -------------------------------------------------------------------
df = orders.merge(order_items_merged, on="order_id", how="left")
df = df.merge(payments_merged,        on="order_id", how="left")
df = df.merge(reviews_dedup,          on="order_id", how="left")
df = df.merge(products_with_category, on="product_id", how="left")
df = df.merge(sellers,                on="seller_id",  how="left")
df = df.merge(customers,              on="customer_id", how="left")

n_post_merge_before_filters = df.shape[0]
print(f"\nPost-merge order count: {n_post_merge_before_filters}")


Post-merge order count: 99441


### 1.2 Completeness filters and outcome (§3.2–3.3, Table 1)

Two filters define the sample: orders without `order_delivered_customer_date` are removed (2,965),
then orders without a review score (646). The final sample has **95,830 orders**.
The outcome `dissatisfied` is 1 for 1–2-star reviews and 0 for 3–5 stars (12,276 orders, 12.81%).

In [8]:
# -------------------------------------------------------------------
# 7. Completeness filters
#    We need a delivery date to compute timing features (H1/H2) and a
#    review score as the outcome variable.
#    Counts are stored programmatically for later missing-data bounds.
# -------------------------------------------------------------------
n_missing_delivery_date = int(df["order_delivered_customer_date"].isna().sum())
df = df[df["order_delivered_customer_date"].notna()].copy()
n_after_delivery_filter = df.shape[0]
print(
    f"After delivery-date filter: {n_after_delivery_filter}  "
    f"(dropped: {n_missing_delivery_date})"
)

n_missing_review_after_delivery = int(df["review_score"].isna().sum())
df = df[df["review_score"].notna()].copy()
print(
    f"After review-score filter:  {df.shape[0]}  "
    f"(dropped: {n_missing_review_after_delivery})"
)

After delivery-date filter: 96476  (dropped: 2965)
After review-score filter:  95830  (dropped: 646)


In [9]:
# -------------------------------------------------------------------
# 8. Category count check
# -------------------------------------------------------------------
n_categories = df["product_category_name_english"].nunique()
print(f"\nUnique product categories: {n_categories}")
print(f"Missing category (NaN):    {df['product_category_name_english'].isna().sum()}")

# -------------------------------------------------------------------
# 9. Binary outcome: dissatisfied = 1 if review_score ≤ 2
# -------------------------------------------------------------------
df["dissatisfied"] = (df["review_score"] <= 2).astype(int)

n_final_analytical = int(len(df))
n_dissatisfied_final = int(df["dissatisfied"].sum())
dissat_rate = df["dissatisfied"].mean()

print(f"\nDissatisfaction rate: {dissat_rate:.4f} ({dissat_rate * 100:.2f}%)")
print(df["dissatisfied"].value_counts())

sample_flow_df = pd.DataFrame([{
    "post_merge_before_filters": n_post_merge_before_filters,
    "missing_delivery_date": n_missing_delivery_date,
    "after_delivery_filter": n_after_delivery_filter,
    "missing_review_after_delivery_filter": n_missing_review_after_delivery,
    "final_analytical_sample": n_final_analytical,
    "final_dissatisfied": n_dissatisfied_final,
    "final_dissatisfaction_rate": dissat_rate,
}])
sample_flow_df.to_csv(f"{OUT_DIR}/h1_sample_flow.csv", index=False)

# -------------------------------------------------------------------
# 10. Save merged dataset
# -------------------------------------------------------------------
out_path = f"{DATA_DIR}/orders_merged.csv"
df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Final shape: {df.shape}")


Unique product categories: 73
Missing category (NaN):    1350

Dissatisfaction rate: 0.1281 (12.81%)
dissatisfied
0    83554
1    12276
Name: count, dtype: int64



Saved: orders_merged.csv
Final shape: (95830, 39)


### 1.3 Feature engineering (§3.4, §4.1, Table K2)

- **Promised-date deviation** (`delivery_delay_days`) is measured in **calendar days**, because the
  promised date is stored at midnight: positive = late, 0 = on the promised day, negative = early.
- **Promised delivery window** (`estimated_delivery_days`): calendar days from purchase to promised date.
- **Days late / days early** (`delay_positive`, `delay_negative`) split the deviation for H2a.
- Commercial, product and geographic predictors; missing catalogue information is set to 0 and
  flagged with `has_missing_product_info`.
- The 73 source categories are mapped to **six category groups** (`category_group_map`, Appendix J.7).

In [10]:
# ============================================================
# DATA PREPARATION — Step 2: Feature Engineering
# ============================================================

print("\n" + "=" * 60)
print("DATA PREPARATION — Step 2: Feature Engineering")
print("=" * 60)

df = pd.read_csv(f"{DATA_DIR}/orders_merged.csv")
print(f"Starting shape: {df.shape}")

# -------------------------------------------------------------------
# 1. Parse date columns
# -------------------------------------------------------------------
date_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    df[c] = pd.to_datetime(df[c])

# -------------------------------------------------------------------
# 2. Delivery timing variables
# -------------------------------------------------------------------

# Actual end-to-end delivery duration.
# Both purchase and actual delivery contain genuine timestamps,
# so fractional days are meaningful here.
df["delivery_time_days"] = (
    df["order_delivered_customer_date"]
    - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400


# Promised delivery window.
# The estimated delivery field represents a promised calendar date
# rather than a meaningful time-of-day, so use calendar-day differences.
df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"].dt.normalize()
    - df["order_purchase_timestamp"].dt.normalize()
).dt.days


# Signed deviation from the promised delivery DATE.
# positive = delivered on a later calendar day
# zero     = delivered on the promised calendar day
# negative = delivered on an earlier calendar day
df["delivery_delay_days"] = (
    df["order_delivered_customer_date"].dt.normalize()
    - df["order_estimated_delivery_date"].dt.normalize()
).dt.days


# Binary late indicator
df["is_late"] = (
    df["delivery_delay_days"] > 0
).astype(int)


# Piecewise decomposition used later for H2 asymmetry analysis
df["delay_positive"] = (
    df["delivery_delay_days"].clip(lower=0)
)

df["delay_negative"] = (
    -df["delivery_delay_days"]
).clip(lower=0)
# -------------------------------------------------------------------
# 3. Price / payment variables
# -------------------------------------------------------------------
df = df.rename(columns={"total_price": "order_price", "total_freight": "freight_value"})

# Freight-to-price ratio captures the relative shipping cost burden;
# for zero-price orders we set the ratio to 0 rather than infinity
df["freight_ratio"] = df["freight_value"] / df["order_price"].replace(0, np.nan)
df["freight_ratio"] = df["freight_ratio"].fillna(0)

# -------------------------------------------------------------------
# 4. Product category grouping → 6 coarse groups
# -------------------------------------------------------------------
category_group_map = {
    # electronics
    "electronics": "electronics", "computers": "electronics",
    "computers_accessories": "electronics", "audio": "electronics",
    "tablets_printing_image": "electronics", "telephony": "electronics",
    "fixed_telephony": "electronics", "consoles_games": "electronics",
    "cine_photo": "electronics", "pc_gamer": "electronics",
    "dvds_blu_ray": "electronics", "cds_dvds_musicals": "electronics",
    "small_appliances": "electronics", "small_appliances_home_oven_and_coffee": "electronics",
    "home_appliances": "electronics", "home_appliances_2": "electronics",
    # health & beauty
    "health_beauty": "health_beauty", "perfumery": "health_beauty",
    "diapers_and_hygiene": "health_beauty", "baby": "health_beauty",
    # furniture & home
    "furniture_bedroom": "furniture_home", "furniture_decor": "furniture_home",
    "furniture_living_room": "furniture_home", "furniture_mattress_and_upholstery": "furniture_home",
    "office_furniture": "furniture_home", "bed_bath_table": "furniture_home",
    "housewares": "furniture_home", "home_comfort_2": "furniture_home",
    "home_confort": "furniture_home", "home_construction": "furniture_home",
    "kitchen_dining_laundry_garden_furniture": "furniture_home",
    "kitchen_portables_and_food_preparers": "furniture_home", "la_cuisine": "furniture_home",
    "garden_tools": "furniture_home", "costruction_tools_garden": "furniture_home",
    "costruction_tools_tools": "furniture_home", "construction_tools_construction": "furniture_home",
    "construction_tools_lights": "furniture_home", "construction_tools_safety": "furniture_home",
    "air_conditioning": "furniture_home", "flowers": "furniture_home",
    # clothing & fashion
    "fashio_female_clothing": "clothing_fashion", "fashion_bags_accessories": "clothing_fashion",
    "fashion_childrens_clothes": "clothing_fashion", "fashion_male_clothing": "clothing_fashion",
    "fashion_shoes": "clothing_fashion", "fashion_sport": "clothing_fashion",
    "fashion_underwear_beach": "clothing_fashion", "luggage_accessories": "clothing_fashion",
    "watches_gifts": "clothing_fashion",
    # sports & leisure
    "sports_leisure": "sports_leisure", "toys": "sports_leisure", "music": "sports_leisure",
    "musical_instruments": "sports_leisure", "party_supplies": "sports_leisure",
    "christmas_supplies": "sports_leisure", "art": "sports_leisure",
    "arts_and_craftmanship": "sports_leisure", "pet_shop": "sports_leisure",
    "cool_stuff": "sports_leisure",
    # other
    "agro_industry_and_commerce": "other", "auto": "other",
    "books_general_interest": "other", "books_imported": "other", "books_technical": "other",
    "drinks": "other", "food": "other", "food_drink": "other",
    "industry_commerce_and_business": "other", "market_place": "other",
    "security_and_services": "other", "signaling_and_security": "other", "stationery": "other",
}

df["category_group"] = df["product_category_name_english"].map(category_group_map)
df["category_group"] = df["category_group"].fillna("other")  # unmapped → other

print("\nCategory group distribution:")
print(df["category_group"].value_counts())

# Label-encode the group for models that require numeric inputs
le_category = LabelEncoder()
df["category_group_encoded"] = le_category.fit_transform(df["category_group"])
print("\nCategory group encoding:")
for code, label in enumerate(le_category.classes_):
    print(f"  {code}: {label}")

# -------------------------------------------------------------------
# 5. Product description length and photo count
#    ~1,350 orders lack product metadata entirely; this missingness
#    itself correlates with higher dissatisfaction (~16.8% vs 12.8%),
#    so we flag it rather than imputing with the median.
# -------------------------------------------------------------------
df["has_missing_product_info"] = df["product_description_lenght"].isna().astype(int)
df["product_description_lenght"] = df["product_description_lenght"].fillna(0)
df["product_photos_qty"]         = df["product_photos_qty"].fillna(0)

# -------------------------------------------------------------------
# 6. Geography: same-state indicator
# -------------------------------------------------------------------
df["same_state"] = (df["customer_state"] == df["seller_state"]).astype(int)

# -------------------------------------------------------------------
# 7. Select final feature columns and save
# -------------------------------------------------------------------
feature_cols = [
    "order_id", "seller_id", "dissatisfied", "review_score",
    "order_purchase_timestamp",
    "delivery_time_days", "estimated_delivery_days", "delivery_delay_days",
    "is_late", "delay_positive", "delay_negative",
    "order_price", "freight_value", "freight_ratio",
    "payment_type", "installment_count", "payment_value",
    "category_group", "category_group_encoded",
    "product_description_lenght", "product_photos_qty", "has_missing_product_info",
    "customer_state", "seller_state", "same_state",
]

df_features = df[feature_cols].copy()

# Fill the single order with no payment record
df_features["payment_type"]     = df_features["payment_type"].fillna("not_defined")
df_features["installment_count"] = df_features["installment_count"].fillna(0)
df_features["payment_value"]     = df_features["payment_value"].fillna(0)

print("\nDelivery variable summary statistics:")
print(df_features[["delivery_time_days", "estimated_delivery_days",
                    "delivery_delay_days", "delay_positive", "delay_negative"]].describe())

print(f"\nLate delivery rate (is_late): {df_features['is_late'].mean():.4f}")

print("\nMissing value check:")
print(df_features.isna().sum()[df_features.isna().sum() > 0])

out_path = f"{DATA_DIR}/orders_features.csv"
df_features.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Final shape: {df_features.shape}")


DATA PREPARATION — Step 2: Feature Engineering


Starting shape: (95830, 39)

Category group distribution:
category_group
furniture_home      28819
sports_leisure      17382
electronics         16591
health_beauty       14394
other                9890
clothing_fashion     8754
Name: count, dtype: int64

Category group encoding:
  0: clothing_fashion
  1: electronics
  2: furniture_home
  3: health_beauty
  4: other
  5: sports_leisure

Delivery variable summary statistics:
       delivery_time_days  estimated_delivery_days  delivery_delay_days  \
count        95830.000000             95830.000000         95830.000000   
mean            12.517256                24.369905           -11.914067   
std              9.460892                 8.758864            10.113511   
min              0.533414                 3.000000          -147.000000   
25%              6.757920                19.000000           -17.000000   
50%             10.206227                24.000000           -12.000000   
75%             15.664106                29.00


Saved: orders_features.csv
Final shape: (95830, 25)


### 1.4 Sample composition (§3.2)

Share of orders with more than one item, product, seller or category group. Category groups are
assigned with the same mapping as above (a missing category counts as "other"), so the 453
multi-category orders are those whose items fall into more than one of the six groups.

In [11]:
# ============================================================
# SAMPLE COMPOSITION OF MULTI-ITEM ORDERS (Section 3.2)
# ============================================================
sample_order_ids = set(df["order_id"])

items_in_sample = (
    order_items[order_items["order_id"].isin(sample_order_ids)]
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(category_translation, on="product_category_name", how="left")
)
items_in_sample["category_group"] = (
    items_in_sample["product_category_name_english"]
    .map(category_group_map)
    .fillna("other")          # same rule as the order-level category group
)

items_by_order = items_in_sample.groupby("order_id")
composition = pd.DataFrame([
    {"order_type": "more than one item",
     "n_orders": int((items_by_order.size() > 1).sum())},
    {"order_type": "more than one distinct product",
     "n_orders": int((items_by_order["product_id"].nunique() > 1).sum())},
    {"order_type": "more than one seller",
     "n_orders": int((items_by_order["seller_id"].nunique() > 1).sum())},
    {"order_type": "more than one category group",
     "n_orders": int((items_by_order["category_group"].nunique() > 1).sum())},
])
composition["share_of_sample_pct"] = (100 * composition["n_orders"] / len(sample_order_ids)).round(2)

print(f"Orders in sample: {len(sample_order_ids):,}")
print(composition.to_string(index=False))
composition.to_csv(f"{OUT_DIR}/data_sample_composition.csv", index=False)

Orders in sample: 95,830
                    order_type  n_orders  share_of_sample_pct
            more than one item      9529                 9.94
more than one distinct product      3160                 3.30
          more than one seller      1261                 1.32
  more than one category group       453                 0.47


### 1.5 Review-record and exclusion diagnostics (§3.2)

- **Shared reviews:** some customers review several orders at once, so the same review is linked to
  more than one order. These 1,008 orders are kept (each has its own delivery timing); their
  dissatisfaction rate is compared with the full sample (22.3% vs 12.8%).
- **Excluded orders:** among the 2,843 orders removed for a missing delivery date that still have a
  review score, 78.05% are dissatisfied, and most are shipped, canceled or unavailable. The sample is
  therefore one of *fulfilled* orders.

In [12]:
# ============================================================
# REVIEW-RECORD AND EXCLUSION DIAGNOSTICS (Section 3.2)
# ============================================================
latest_review = (
    order_reviews.sort_values("review_creation_date")
    .drop_duplicates("order_id", keep="last")
)

# 1. Orders whose (most recent) review is shared with at least one other order
review_link = latest_review[["order_id", "review_id"]].copy()
review_link["shared_review"] = review_link["review_id"].duplicated(keep=False)

shared_check = df[["order_id", "dissatisfied"]].merge(
    review_link, on="order_id", how="left", validate="one_to_one"
)
n_shared = int(shared_check["shared_review"].sum())
print(f"Orders in sample sharing their review with another order: {n_shared:,} "
      f"({n_shared / len(shared_check):.2%})")
print(f"Dissatisfaction rate, shared-review orders: "
      f"{shared_check.loc[shared_check['shared_review'], 'dissatisfied'].mean():.1%}")
print(f"Dissatisfaction rate, full sample:          {shared_check['dissatisfied'].mean():.1%}")

# 2. Orders excluded because the delivery date is missing, but with an observed review
excluded = orders.merge(latest_review[["order_id", "review_score"]], on="order_id", how="left")
excluded = excluded[
    excluded["order_delivered_customer_date"].isna() & excluded["review_score"].notna()
]
print(f"\nExcluded orders (no delivery date) with a review score: {len(excluded):,}")
print(f"  of which dissatisfied (1-2 stars): {(excluded['review_score'] <= 2).sum():,} "
      f"({(excluded['review_score'] <= 2).mean():.2%})")
print("\nOrder status of these excluded orders:")
print(excluded["order_status"].value_counts().to_string())

Orders in sample sharing their review with another order: 1,008 (1.05%)
Dissatisfaction rate, shared-review orders: 22.3%
Dissatisfaction rate, full sample:          12.8%

Excluded orders (no delivery date) with a review score: 2,843
  of which dissatisfied (1-2 stars): 2,219 (78.05%)

Order status of these excluded orders:
order_status
shipped        1032
canceled        599
unavailable     595
invoiced        309
processing      295
delivered         8
created           3
approved          2


### 1.6 Calendar-day timing definition check (Appendix J.1, Table J1)

Rebuilds the timing variables independently and shows the effect of using calendar days: 1,280 orders
delivered on the promised day would be counted as late under a timestamp definition.

In [13]:
# ============================================================
# DELIVERY-TIMING DEFINITION — FINAL SANITY CHECK
# ============================================================

print("\n" + "=" * 72)
print("FINAL DELIVERY-TIMING DEFINITION CHECK")
print("=" * 72)

raw_check = pd.read_csv(
    f"{DATA_DIR}/orders_merged.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

# Keep exactly the analytical sample
check = df_features[
    [
        "order_id",
        "delivery_time_days",
        "estimated_delivery_days",
        "delivery_delay_days",
        "is_late",
        "delay_positive",
        "delay_negative",
    ]
].merge(
    raw_check[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
    ],
    on="order_id",
    how="left",
)


# ------------------------------------------------------------
# 1. Does estimated delivery contain meaningful time-of-day?
# ------------------------------------------------------------
estimated_non_midnight = (
    (
        check["order_estimated_delivery_date"].dt.hour != 0
    )
    |
    (
        check["order_estimated_delivery_date"].dt.minute != 0
    )
    |
    (
        check["order_estimated_delivery_date"].dt.second != 0
    )
).sum()

print(
    f"Estimated-delivery timestamps not at midnight: "
    f"{estimated_non_midnight:,}"
)


# ------------------------------------------------------------
# 2. Recompute calendar-day variables independently
# ------------------------------------------------------------
calendar_delay_check = (
    check["order_delivered_customer_date"].dt.normalize()
    - check["order_estimated_delivery_date"].dt.normalize()
).dt.days

calendar_window_check = (
    check["order_estimated_delivery_date"].dt.normalize()
    - check["order_purchase_timestamp"].dt.normalize()
).dt.days

print(
    "Max stored-vs-recomputed delay difference:",
    np.abs(
        check["delivery_delay_days"]
        - calendar_delay_check
    ).max(),
)

print(
    "Max stored-vs-recomputed promised-window difference:",
    np.abs(
        check["estimated_delivery_days"]
        - calendar_window_check
    ).max(),
)


# ------------------------------------------------------------
# 3. Piecewise H2 decomposition
# ------------------------------------------------------------
reconstructed = (
    check["delay_positive"]
    - check["delay_negative"]
)

print(
    "Max delay reconstruction error:",
    np.abs(
        reconstructed
        - check["delivery_delay_days"]
    ).max(),
)

print(
    "Orders with both positive and negative components > 0:",
    (
        (check["delay_positive"] > 0)
        &
        (check["delay_negative"] > 0)
    ).sum(),
)


# ------------------------------------------------------------
# 4. Late indicator consistency
# ------------------------------------------------------------
print(
    "is_late mismatches:",
    (
        check["is_late"]
        != (
            check["delivery_delay_days"] > 0
        ).astype(int)
    ).sum(),
)


# ------------------------------------------------------------
# 5. Early / on-time / late distribution
# ------------------------------------------------------------
status = np.select(
    [
        check["delivery_delay_days"] < 0,
        check["delivery_delay_days"] == 0,
        check["delivery_delay_days"] > 0,
    ],
    [
        "early",
        "on_time",
        "late",
    ],
    default="unknown",
)

print("\nCalendar-day delivery status:")
print(
    pd.Series(status)
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 6. Compare old timestamp definition with final definition
# ------------------------------------------------------------
old_timestamp_delay = (
    check["order_delivered_customer_date"]
    - check["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

old_late = (
    old_timestamp_delay > 0
)

new_late = (
    check["delivery_delay_days"] > 0
)

print(
    "\nOrders whose late/not-late classification changed:",
    (old_late != new_late).sum(),
)

print(
    "Correlation old vs new delay:",
    np.corrcoef(
        old_timestamp_delay,
        check["delivery_delay_days"],
    )[0, 1],
)


# ------------------------------------------------------------
# 7. Basic impossible-value checks
# ------------------------------------------------------------
print(
    "\nNegative actual delivery durations:",
    (check["delivery_time_days"] < 0).sum(),
)

print(
    "Negative promised delivery windows:",
    (check["estimated_delivery_days"] < 0).sum(),
)

print(
    "Missing values in timing variables:",
    check[
        [
            "delivery_time_days",
            "estimated_delivery_days",
            "delivery_delay_days",
            "delay_positive",
            "delay_negative",
        ]
    ].isna().sum().sum(),
)


FINAL DELIVERY-TIMING DEFINITION CHECK


Estimated-delivery timestamps not at midnight: 0
Max stored-vs-recomputed delay difference: 0
Max stored-vs-recomputed promised-window difference: 0
Max delay reconstruction error: 0
Orders with both positive and negative components > 0: 0
is_late mismatches: 0

Calendar-day delivery status:
early      88168
late        6382
on_time     1280

Orders whose late/not-late classification changed: 1280
Correlation old vs new delay: 0.999768214995842

Negative actual delivery durations: 0
Negative promised delivery windows: 0
Missing values in timing variables: 0


### 1.7 Descriptive statistics (§3.4, Table K1)

In [14]:
# ============================================================
# DESCRIPTIVE STATISTICS (Table K1)
# ============================================================
late_only = df_features.loc[df_features["delay_positive"] > 0, "delay_positive"]
early_only = df_features.loc[df_features["delay_negative"] > 0, "delay_negative"]

k1_sources = {
    "Actual delivery time": df_features["delivery_time_days"],
    "Promised delivery window": df_features["estimated_delivery_days"],
    "Promised-date deviation": df_features["delivery_delay_days"],
    "Days late (all orders)": df_features["delay_positive"],
    f"Days late (late only, n = {len(late_only):,})": late_only,
    "Days early (all orders)": df_features["delay_negative"],
    f"Days early (early only, n = {len(early_only):,})": early_only,
    "Order value (BRL)": df_features["order_price"],
    "Freight cost (BRL)": df_features["freight_value"],
    "Late-delivery indicator (proportion)": df_features["is_late"],
    "Dissatisfaction (proportion)": df_features["dissatisfied"],
}
table_k1 = pd.DataFrame({
    name: {"Mean": s.mean(), "Std Dev": s.std(), "Median": s.median(), "Max": s.max()}
    for name, s in k1_sources.items()
}).T.round(4)
print(table_k1.to_string())
table_k1.to_csv(f"{OUT_DIR}/table_k1_descriptives.csv")

# Table 2: orders and dissatisfaction by category group
table_2 = (
    df_features.groupby("category_group")["dissatisfied"]
    .agg(n_orders="size", n_dissatisfied="sum", dissatisfaction_rate_pct="mean")
    .sort_values("n_orders", ascending=False)
)
table_2["dissatisfaction_rate_pct"] = (100 * table_2["dissatisfaction_rate_pct"]).round(2)
print("\nTable 2 — orders and dissatisfaction by category group")
print(table_2.to_string())

                                          Mean   Std Dev   Median         Max
Actual delivery time                   12.5173    9.4609  10.2062    208.3518
Promised delivery window               24.3699    8.7589  24.0000    156.0000
Promised-date deviation               -11.9141   10.1135 -12.0000    188.0000
Days late (all orders)                  0.7005    4.5611   0.0000    188.0000
Days late (late only, n = 6,382)       10.5186   14.4615   7.0000    188.0000
Days early (all orders)                12.6146    7.9879  12.0000    147.0000
Days early (early only, n = 88,168)    13.7108    7.3702  13.0000    147.0000
Order value (BRL)                     136.7984  207.7828  86.2500  13440.0000
Freight cost (BRL)                     22.7625   21.5221  17.1600   1794.9600
Late-delivery indicator (proportion)    0.0666    0.2493   0.0000      1.0000
Dissatisfaction (proportion)            0.1281    0.3342   0.0000      1.0000

Table 2 — orders and dissatisfaction by category group
        

### 1.8 Category grouping and composition of "Other" (Appendix J.7, Table J7)

Verifies that every non-missing source category is mapped to one of the five named groups or
deliberately assigned to "other" (8,540 explicit + 1,350 missing-category orders).

In [15]:
# ============================================================
# CATEGORY-GROUP "OTHER" COMPOSITION DIAGNOSTIC
# ============================================================

print("\n" + "=" * 72)
print('CATEGORY-GROUP "OTHER" COMPOSITION DIAGNOSTIC')
print("=" * 72)

# Reload the correct dataset independently.
# Do NOT use the current notebook's df object.
category_audit = pd.read_csv(
    f"{DATA_DIR}/orders_merged.csv"
)

category_col = "product_category_name_english"

# Safety check
if category_col not in category_audit.columns:
    raise RuntimeError(
        f"{category_col} is not present in orders_merged.csv. "
        f"Available columns: {category_audit.columns.tolist()}"
    )


# ------------------------------------------------------------
# 1. IDENTIFY HOW EACH ORIGINAL CATEGORY IS HANDLED
# ------------------------------------------------------------

# Categories deliberately assigned to "other"
explicit_other_categories = {
    category
    for category, group in category_group_map.items()
    if group == "other"
}

# All categories explicitly represented in the mapping
mapped_categories = set(
    category_group_map.keys()
)


def classify_category_source(value):

    if pd.isna(value):
        return "missing_category"

    if value in explicit_other_categories:
        return "explicit_other"

    if value not in mapped_categories:
        return "unmapped_nonmissing"

    return "mapped_to_named_group"


category_audit["category_source_type"] = (
    category_audit[category_col]
    .apply(classify_category_source)
)


# ------------------------------------------------------------
# 2. OVERALL COMPOSITION
# ------------------------------------------------------------

composition = (
    category_audit["category_source_type"]
    .value_counts(dropna=False)
    .rename_axis("category_source_type")
    .reset_index(name="n_orders")
)

composition["share"] = (
    composition["n_orders"]
    / len(category_audit)
)

print(
    "\n=== Overall category-source composition ==="
)

print(
    composition.to_string(index=False)
)


# ------------------------------------------------------------
# 3. WHAT ENDS UP IN FINAL "OTHER"?
# ------------------------------------------------------------

other_types = [
    "explicit_other",
    "missing_category",
    "unmapped_nonmissing",
]

other_mask = (
    category_audit["category_source_type"]
    .isin(other_types)
)

other_composition = (
    category_audit.loc[
        other_mask,
        "category_source_type"
    ]
    .value_counts()
    .rename_axis("source")
    .reset_index(name="n_orders")
)

other_composition["share_of_other"] = (
    other_composition["n_orders"]
    / other_composition["n_orders"].sum()
)

print(
    '\n=== Composition of final "other" group ==='
)

print(
    other_composition.to_string(index=False)
)


# ------------------------------------------------------------
# 4. EXPLICIT CATEGORIES DELIBERATELY MAPPED TO OTHER
# ------------------------------------------------------------

explicit_other_detail = (
    category_audit.loc[
        category_audit["category_source_type"]
        == "explicit_other",
        category_col,
    ]
    .value_counts()
    .rename_axis("product_category")
    .reset_index(name="n_orders")
)

print(
    "\n=== Explicit categories deliberately mapped to other ==="
)

print(
    explicit_other_detail.to_string(index=False)
)


# ------------------------------------------------------------
# 5. UNMAPPED NON-MISSING CATEGORIES
# ------------------------------------------------------------

unmapped_detail = (
    category_audit.loc[
        category_audit["category_source_type"]
        == "unmapped_nonmissing",
        category_col,
    ]
    .value_counts()
    .rename_axis("product_category")
    .reset_index(name="n_orders")
)

print(
    "\n=== Non-missing categories absent from category_group_map ==="
)

if len(unmapped_detail) == 0:
    print("NONE")
else:
    print(
        unmapped_detail.to_string(index=False)
    )


# ------------------------------------------------------------
# 6. SIMPLE COUNTS
# ------------------------------------------------------------

n_other_expected = int(
    other_mask.sum()
)

n_missing = int(
    category_audit[category_col]
    .isna()
    .sum()
)

n_unmapped = int(
    (
        category_audit["category_source_type"]
        == "unmapped_nonmissing"
    ).sum()
)

print(
    '\nOrders that should become category_group="other": '
    f'{n_other_expected:,}'
)

print(
    f"Missing original category: {n_missing:,}"
)

print(
    f"Unmapped non-missing categories: {n_unmapped:,}"
)


CATEGORY-GROUP "OTHER" COMPOSITION DIAGNOSTIC



=== Overall category-source composition ===
 category_source_type  n_orders    share
mapped_to_named_group     85940 0.896796
       explicit_other      8540 0.089116
     missing_category      1350 0.014087

=== Composition of final "other" group ===
          source  n_orders  share_of_other
  explicit_other      8540        0.863498
missing_category      1350        0.136502

=== Explicit categories deliberately mapped to other ===
              product_category  n_orders
                          auto      3774
                    stationery      2234
        books_general_interest       488
                          food       431
                        drinks       284
                  market_place       266
               books_technical       254
industry_commerce_and_business       226
                    food_drink       218
    agro_industry_and_commerce       177
        signaling_and_security       137
                books_imported        49
         security_and_servi

---
## Part 2 — Phase 1: Machine-learning classification (H1)
*Thesis §4.2 and §5.1. H1: delivery timing is the strongest structured predictor of dissatisfaction,
and this holds across product categories.*

**Design.** The promised-date deviation is the timing predictor and the promised delivery window is
a control; other timing variables (actual delivery time, late indicator, days late/early) are
excluded because they duplicate the same information. After one-hot encoding the model has 67
predictors. The data are split once into a stratified 80% training sample (76,664) and a 20% holdout
(19,166). All tuning uses five-fold stratified CV on the training sample only (ROC-AUC); the holdout
is used once, for final evaluation.

In [16]:
# ============================================================
# H1: PREDICTIVE IMPORTANCE OF DELIVERY TIMING
# ============================================================
# Research goal:
#   Test whether realised delivery delay is the strongest predictor
#   of customer dissatisfaction among the included order-level
#   predictors, and whether this pattern is consistent across
#   product-category groups.
#
# Delivery-timing operationalisation:
#   - delivery_delay_days:
#       focal H1 predictor. Calendar-day deviation between actual
#       and promised delivery date:
#         positive = delivered after the promised date
#         zero     = delivered on the promised date
#         negative = delivered before the promised date
#
#   - estimated_delivery_days:
#       promised delivery-window length, included as an ex-ante
#       delivery-expectation control.
#
#   Deterministic / overlapping timing representations
#   (delivery_time_days, is_late, delay_positive, delay_negative)
#   are excluded from H1 to avoid representing the same underlying
#   timing information multiple times.
#
#   delay_positive and delay_negative are reserved for H2, where
#   asymmetry between lateness and earliness is explicitly tested.
#
# Methodological structure:
#   1. Create a stratified 80/20 primary holdout split.
#   2. Tune LR, RF, and XGBoost only inside the 80% training sample
#      using 5-fold stratified CV and ROC-AUC.
#   3. Freeze the selected specifications and evaluate them on the
#      20% holdout sample.
#   4. Select the better-performing tree ensemble (RF vs XGBoost)
#      using training-CV ROC-AUC for TreeSHAP explanation.
#   5. Evaluate H1 using:
#        a) global TreeSHAP,
#        b) category-specific TreeSHAP,
#        c) comparative predictor-level drop-column ablation,
#        d) category-specific predictor ablation.
#   6. Robustness:
#        seller-grouped CV,
#        alternative random split ratios,
#        temporal holdout,
#        temporal predictor ablation,
#        and missing-delivery-date sample sensitivity.
#
# Important:
#   - Hyperparameter selection is based only on training-sample CV.
#   - H1 concerns predictive importance, not late-vs-early asymmetry.
#   - "Strongest" is assessed primarily through comparative
#     predictor ablation, with SHAP as complementary evidence.
# ============================================================

print("\n" + "=" * 72)
print("H1: PREDICTIVE IMPORTANCE OF DELIVERY TIMING")
print("=" * 72)

df = pd.read_csv(
    f"{DATA_DIR}/orders_features.csv",
    parse_dates=["order_purchase_timestamp"],
)

print(f"Data shape: {df.shape}")

target = "dissatisfied"

categorical_cols = [
    "payment_type",
    "customer_state",
    "seller_state",
    "category_group",
]

# ------------------------------------------------------------
# H1 feature specification
# ------------------------------------------------------------
# Delivery timing is deliberately represented parsimoniously:
#   delivery_delay_days     = realised deviation from promise
#   estimated_delivery_days = promised delivery window
#
# The following engineered timing variables are NOT included in H1:
#   delivery_time_days
#   is_late
#   delay_positive
#   delay_negative
#
# They contain deterministic or strongly overlapping representations
# of the timing information above and are unnecessary for testing H1.
# delay_positive and delay_negative remain available in the dataset
# for the separate H2 asymmetry analysis.
# ------------------------------------------------------------

numeric_cols = [
    # Delivery timing
    "delivery_delay_days",
    "estimated_delivery_days",

    # Commercial / order characteristics
    "order_price",
    "freight_value",
    "freight_ratio",

    # Payment characteristics
    "installment_count",
    "payment_value",

    # Product characteristics
    "product_description_lenght",
    "product_photos_qty",
    "has_missing_product_info",

    # Geography
    "same_state",
]

if "seller_id" not in df.columns:
    raise ValueError(
        "seller_id is missing from orders_features.csv. "
        "Add seller_id to feature_cols in the feature-engineering section "
        "and regenerate orders_features.csv before running H1."
    )

X = df[categorical_cols + numeric_cols].copy()
y = df[target].astype(int).copy()
seller_groups = df["seller_id"].copy()

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
)

print(f"Feature count after one-hot encoding: {X.shape[1]}")
print(f"Dissatisfaction rate: {y.mean():.4f}")

print("\nH1 timing variables:")
print("  delivery_delay_days")
print("  estimated_delivery_days")


H1: PREDICTIVE IMPORTANCE OF DELIVERY TIMING
Data shape: (95830, 25)


Feature count after one-hot encoding: 67
Dissatisfaction rate: 0.1281

H1 timing variables:
  delivery_delay_days
  estimated_delivery_days


### 2.1 Holdout split and Logistic Regression tuning (Appendix G.1)

Logistic Regression is tuned over `C` and class weighting, with standardization fitted inside each fold.

In [17]:
# ------------------------------------------------------------
# 1. PRIMARY STRATIFIED 80/20 HOLDOUT
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("\nPrimary holdout:")
print(f"  Train n={len(y_train):,}, dissatisfied={y_train.mean():.4f}")
print(f"  Test  n={len(y_test):,}, dissatisfied={y_test.mean():.4f}")

# Hyperparameter tuning occurs ONLY within X_train / y_train.
cv_tune = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)


# ------------------------------------------------------------
# 2. LOGISTIC REGRESSION — TRAINING-ONLY CV TUNING
# ------------------------------------------------------------
# Numeric variables are standardized INSIDE each CV fold.
# Dummy variables are passed through unchanged.
#
# L2 regularization is used as the conventional predictive LR
# specification. C controls inverse regularization strength and is
# searched over logarithmically spaced orders of magnitude.
#
# Class weighting is not imposed a priori; both unweighted and
# balanced specifications are evaluated by training-only CV.
# ------------------------------------------------------------

lr_preprocess = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_cols),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

lr_pipeline = Pipeline(
    steps=[
        ("preprocess", lr_preprocess),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                max_iter=2000,
            ),
        ),
    ]
)

lr_param_grid = {
    "model__C": np.logspace(-3, 3, 7),
    "model__class_weight": [None, "balanced"],
}

print("\nTuning Logistic Regression on training data only...")

lr_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    scoring="roc_auc",
    cv=cv_tune,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

lr_search.fit(X_train, y_train)

best_lr = lr_search.best_estimator_

lr_stage1_best_cv_auc = lr_search.best_score_
lr_stage1_best_params = lr_search.best_params_

print(f"  Best CV AUC: {lr_stage1_best_cv_auc:.6f}")
print(f"  Best params: {lr_stage1_best_params}")


# ------------------------------------------------------------
# SAVE COMPLETE LR TUNING RESULTS
# ------------------------------------------------------------
lr_tuning_df = (
    pd.DataFrame(lr_search.cv_results_)[
        [
            "param_model__C",
            "param_model__class_weight",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .sort_values(
        ["rank_test_score", "mean_test_score"],
        ascending=[True, False],
    )
)

lr_tuning_df.to_csv(
    f"{OUT_DIR}/h1_tuning_logistic.csv",
    index=False,
)

print("\nTop LR specifications:")
print(lr_tuning_df.head(10).to_string(index=False))


# ------------------------------------------------------------
# BOUNDARY DIAGNOSTIC
# ------------------------------------------------------------
tested_C = np.array(lr_param_grid["model__C"], dtype=float)
best_C = float(lr_stage1_best_params["model__C"])

if np.isclose(best_C, tested_C.min()) or np.isclose(best_C, tested_C.max()):
    print(
        "\n[CHECK] Best LR C lies on the boundary of the tested grid. "
        "Consider a targeted expansion before freezing the LR specification."
    )
else:
    print(
        "\nLR C optimum is interior to the tested grid; "
        "no boundary-driven expansion is required."
    )


Primary holdout:
  Train n=76,664, dissatisfied=0.1281
  Test  n=19,166, dissatisfied=0.1281

Tuning Logistic Regression on training data only...


  Best CV AUC: 0.713517
  Best params: {'model__C': np.float64(1.0), 'model__class_weight': 'balanced'}

Top LR specifications:
 param_model__C param_model__class_weight  mean_test_score  std_test_score  rank_test_score
          1.000                  balanced         0.713517        0.005291                1
        100.000                  balanced         0.713512        0.005289                2
         10.000                  balanced         0.713491        0.005335                3
       1000.000                  balanced         0.713459        0.005260                4
          0.100                  balanced         0.713419        0.005304                5
          0.010                  balanced         0.712409        0.005209                6
          0.001                  balanced         0.708721        0.004316                7
        100.000                      None         0.707353        0.005606                8
          1.000                      None   

In [18]:
# ------------------------------------------------------------
# 2B. FREEZE FINAL LOGISTIC REGRESSION SPECIFICATION
# ------------------------------------------------------------
# The Stage-1 optimum is interior to the prespecified C grid,
# so no boundary-driven grid expansion is required.

FINAL_LR_C = float(
    lr_stage1_best_params["model__C"]
)

FINAL_LR_CLASS_WEIGHT = (
    lr_stage1_best_params["model__class_weight"]
)

FINAL_LR_CV_AUC = (
    lr_stage1_best_cv_auc
)

FINAL_LR_PARAMS = {
    "C": FINAL_LR_C,
    "class_weight": FINAL_LR_CLASS_WEIGHT,
    "penalty": "l2",
    "solver": "lbfgs",
}

# GridSearchCV already refit the winning specification
# on the full training sample because refit=True.
final_lr = lr_search.best_estimator_

print("\nFinal Logistic Regression specification frozen:")
print(f"  Training CV AUC: {FINAL_LR_CV_AUC:.6f}")
print(f"  Parameters: {FINAL_LR_PARAMS}")


Final Logistic Regression specification frozen:
  Training CV AUC: 0.713517
  Parameters: {'C': 1.0, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}


### 2.2 Random Forest tuning (Appendix G.1, Tables G1a–G1b)

Initial grid, then a targeted boundary check because the first optimum was on the edge of the grid.
Final: 1,000 trees, no maximum depth, `min_samples_leaf = 8`, balanced class weights.

In [19]:
# ------------------------------------------------------------
# 3. RANDOM FOREST — TRAINING-ONLY CV TUNING
# ------------------------------------------------------------

rf_base = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1,
)

rf_param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [10, 15, None],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": [None, "balanced"],
}

print("\nTuning Random Forest on training data only...")

rf_search = GridSearchCV(
    estimator=rf_base,
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=cv_tune,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

rf_search.fit(X_train, y_train)

rf_stage1_best_cv_auc = rf_search.best_score_
rf_stage1_best_params = rf_search.best_params_

best_rf = rf_search.best_estimator_

print(
    f"  Best CV AUC: "
    f"{rf_stage1_best_cv_auc:.6f}"
)
print(
    f"  Best params: "
    f"{rf_stage1_best_params}"
)

rf_stage1_df = (
    pd.DataFrame(rf_search.cv_results_)[
        [
            "param_n_estimators",
            "param_max_depth",
            "param_min_samples_leaf",
            "param_class_weight",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .sort_values(
        ["rank_test_score", "mean_test_score"],
        ascending=[True, False],
    )
)

rf_stage1_df.to_csv(
    f"{OUT_DIR}/h1_tuning_random_forest_stage1.csv",
    index=False,
)

print("\nTop RF specifications:")
print(
    rf_stage1_df.head(10).to_string(index=False)
)


Tuning Random Forest on training data only...


  Best CV AUC: 0.735532
  Best params: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_leaf': 4, 'n_estimators': 500}

Top RF specifications:
 param_n_estimators param_max_depth  param_min_samples_leaf param_class_weight  mean_test_score  std_test_score  rank_test_score
                500            None                       4           balanced         0.735532        0.005665                1
                500            None                       4               None         0.735287        0.002852                2
                500              15                       4               None         0.734906        0.001298                3
                500              15                       2               None         0.734056        0.002483                4
                200            None                       4           balanced         0.733960        0.005707                5
                500              15                       1            

In [20]:
# ------------------------------------------------------------
# 3B. RANDOM FOREST — TARGETED BOUNDARY CHECK
# ------------------------------------------------------------

rf_stage2_param_grid = {
    "n_estimators": [500, 750, 1000],
    "max_depth": [None],
    "min_samples_leaf": [4, 6, 8],
    "class_weight": ["balanced"],
}

print("\nRF Stage 2: targeted boundary check...")

rf_stage2_search = GridSearchCV(
    estimator=rf_base,
    param_grid=rf_stage2_param_grid,
    scoring="roc_auc",
    cv=cv_tune,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

rf_stage2_search.fit(X_train, y_train)

rf_stage2_df = (
    pd.DataFrame(rf_stage2_search.cv_results_)[
        [
            "param_n_estimators",
            "param_max_depth",
            "param_min_samples_leaf",
            "param_class_weight",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .sort_values(
        ["rank_test_score", "mean_test_score"],
        ascending=[True, False],
    )
)

print("\nRF Stage 2 results:")
print(rf_stage2_df.to_string(index=False))

rf_stage2_df.to_csv(
    f"{OUT_DIR}/h1_tuning_random_forest_stage2.csv",
    index=False,
)

print(
    f"\nBest Stage 2 CV AUC: "
    f"{rf_stage2_search.best_score_:.6f}"
)
print(
    f"Best Stage 2 params: "
    f"{rf_stage2_search.best_params_}"
)


RF Stage 2: targeted boundary check...



RF Stage 2 results:
 param_n_estimators param_max_depth  param_min_samples_leaf param_class_weight  mean_test_score  std_test_score  rank_test_score
               1000            None                       8           balanced         0.737385        0.004686                1
                750            None                       8           balanced         0.737306        0.004802                2
               1000            None                       6           balanced         0.737065        0.004681                3
                750            None                       6           balanced         0.736853        0.004678                4
                500            None                       8           balanced         0.736684        0.004431                5
                500            None                       6           balanced         0.736580        0.004676                6
               1000            None                       4           balanc

In [21]:
# ------------------------------------------------------------
# 3C. FREEZE FINAL RANDOM FOREST SPECIFICATION
# ------------------------------------------------------------

FINAL_RF_CV_AUC = rf_stage2_search.best_score_

FINAL_RF_PARAMS = {
    "n_estimators": int(
        rf_stage2_search.best_params_["n_estimators"]
    ),
    "max_depth": rf_stage2_search.best_params_["max_depth"],
    "min_samples_leaf": int(
        rf_stage2_search.best_params_["min_samples_leaf"]
    ),
    "class_weight": rf_stage2_search.best_params_["class_weight"],
}

final_rf = rf_stage2_search.best_estimator_
final_rf.set_params(n_jobs=-1)

print("\nFinal Random Forest specification frozen:")
print(f"  Training CV AUC: {FINAL_RF_CV_AUC:.6f}")
print(f"  Parameters: {FINAL_RF_PARAMS}")


Final Random Forest specification frozen:
  Training CV AUC: 0.737385
  Parameters: {'n_estimators': 1000, 'max_depth': None, 'min_samples_leaf': 8, 'class_weight': 'balanced'}


### 2.3 XGBoost tuning (Appendix G.1)

Class weighting is a tuning choice (none, √ratio, full ratio). Final: 200 trees, depth 6,
learning rate 0.05, `scale_pos_weight = 1`.

In [22]:
# ------------------------------------------------------------
# 4. XGBOOST — TRAINING-ONLY CV TUNING
# ------------------------------------------------------------
# XGBoost is tuned exclusively within the 80% training sample.
#
# ROC-AUC is used as the tuning criterion, consistent with H1's
# predictive-discrimination objective.
#
# Class weighting is not imposed a priori. We compare:
#   1. no weighting,
#   2. intermediate sqrt(class-ratio) weighting,
#   3. full class-ratio weighting.
# ------------------------------------------------------------

class_ratio_train = (
    (y_train == 0).sum()
    / (y_train == 1).sum()
)

sqrt_class_ratio = np.sqrt(
    class_ratio_train
)

xgb_base = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=RANDOM_STATE,
    n_jobs=1,       # avoid nested parallelism inside GridSearchCV
    verbosity=0,
)

xgb_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 6],
    "learning_rate": [0.05, 0.10],
    "scale_pos_weight": [
        1.0,
        sqrt_class_ratio,
        class_ratio_train,
    ],
}

print("\nTuning XGBoost on training data only...")

xgb_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_param_grid,
    scoring="roc_auc",
    cv=cv_tune,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

xgb_search.fit(
    X_train,
    y_train,
)

xgb_stage1_best_cv_auc = (
    xgb_search.best_score_
)

xgb_stage1_best_params = (
    xgb_search.best_params_
)

print(
    f"  Training class ratio: "
    f"{class_ratio_train:.6f}"
)

print(
    f"  sqrt(class ratio):    "
    f"{sqrt_class_ratio:.6f}"
)

print(
    f"  Best CV AUC:          "
    f"{xgb_stage1_best_cv_auc:.6f}"
)

print(
    f"  Best params:          "
    f"{xgb_stage1_best_params}"
)


# ------------------------------------------------------------
# SAVE COMPLETE STAGE-1 TUNING RESULTS
# ------------------------------------------------------------

xgb_stage1_df = (
    pd.DataFrame(
        xgb_search.cv_results_
    )[
        [
            "param_n_estimators",
            "param_max_depth",
            "param_learning_rate",
            "param_scale_pos_weight",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .sort_values(
        [
            "rank_test_score",
            "mean_test_score",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

xgb_stage1_df.to_csv(
    f"{OUT_DIR}/h1_tuning_xgboost_stage1.csv",
    index=False,
)

print(
    "\nTop XGBoost specifications:"
)

print(
    xgb_stage1_df.head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# BOUNDARY DIAGNOSTIC
# ------------------------------------------------------------

best_n_estimators = int(
    xgb_stage1_best_params[
        "n_estimators"
    ]
)

best_max_depth = int(
    xgb_stage1_best_params[
        "max_depth"
    ]
)

best_learning_rate = float(
    xgb_stage1_best_params[
        "learning_rate"
    ]
)

n_estimators_boundary = (
    best_n_estimators
    == max(
        xgb_param_grid[
            "n_estimators"
        ]
    )
)

max_depth_boundary = (
    best_max_depth
    == max(
        xgb_param_grid[
            "max_depth"
        ]
    )
)

learning_rate_boundary = (
    np.isclose(
        best_learning_rate,
        min(
            xgb_param_grid[
                "learning_rate"
            ]
        ),
    )
    or
    np.isclose(
        best_learning_rate,
        max(
            xgb_param_grid[
                "learning_rate"
            ]
        ),
    )
)

print("\nXGBoost boundary diagnostic:")

print(
    f"  n_estimators upper boundary: "
    f"{n_estimators_boundary}"
)

print(
    f"  max_depth upper boundary:    "
    f"{max_depth_boundary}"
)

print(
    f"  learning_rate boundary:      "
    f"{learning_rate_boundary}"
)


Tuning XGBoost on training data only...


  Training class ratio: 6.806130
  sqrt(class ratio):    2.608856
  Best CV AUC:          0.737368
  Best params:          {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200, 'scale_pos_weight': 1.0}

Top XGBoost specifications:
 param_n_estimators  param_max_depth  param_learning_rate  param_scale_pos_weight  mean_test_score  std_test_score  rank_test_score
                200                6                 0.05                1.000000         0.737368        0.002684                1
                100                6                 0.05                1.000000         0.737109        0.002300                2
                200                6                 0.05                2.608856         0.736730        0.002776                3
                300                6                 0.05                1.000000         0.736721        0.003269                4
                100                6                 0.05                2.608856         0.736388    

In [23]:
# ------------------------------------------------------------
# 4B. XGBOOST — TARGETED BOUNDARY CHECK
# ------------------------------------------------------------

xgb_stage2_param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [6, 8],
    "learning_rate": [0.025, 0.05],
    "scale_pos_weight": [1.0],
}

print("\nXGBoost Stage 2: targeted boundary check...")

xgb_stage2_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_stage2_param_grid,
    scoring="roc_auc",
    cv=cv_tune,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

xgb_stage2_search.fit(
    X_train,
    y_train,
)

xgb_stage2_df = (
    pd.DataFrame(xgb_stage2_search.cv_results_)[
        [
            "param_n_estimators",
            "param_max_depth",
            "param_learning_rate",
            "param_scale_pos_weight",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .sort_values(
        ["rank_test_score", "mean_test_score"],
        ascending=[True, False],
    )
)

print("\nXGBoost Stage 2 results:")
print(xgb_stage2_df.to_string(index=False))

xgb_stage2_df.to_csv(
    f"{OUT_DIR}/h1_tuning_xgboost_stage2.csv",
    index=False,
)

print(
    f"\nBest Stage 2 CV AUC: "
    f"{xgb_stage2_search.best_score_:.6f}"
)

print(
    f"Best Stage 2 params: "
    f"{xgb_stage2_search.best_params_}"
)


XGBoost Stage 2: targeted boundary check...



XGBoost Stage 2 results:
 param_n_estimators  param_max_depth  param_learning_rate  param_scale_pos_weight  mean_test_score  std_test_score  rank_test_score
                200                6                0.050                     1.0         0.737368        0.002684                1
                300                6                0.025                     1.0         0.737337        0.002789                2
                200                6                0.025                     1.0         0.737326        0.002550                3
                200                8                0.025                     1.0         0.737098        0.004250                4
                500                6                0.025                     1.0         0.737021        0.003055                5
                300                6                0.050                     1.0         0.736721        0.003269                6
                300                8              

In [24]:
# ------------------------------------------------------------
# 4C. FREEZE FINAL XGBOOST SPECIFICATION
# ------------------------------------------------------------

FINAL_XGB_CV_AUC = xgb_stage2_search.best_score_

FINAL_XGB_PARAMS = {
    "n_estimators": int(
        xgb_stage2_search.best_params_["n_estimators"]
    ),
    "max_depth": int(
        xgb_stage2_search.best_params_["max_depth"]
    ),
    "learning_rate": float(
        xgb_stage2_search.best_params_["learning_rate"]
    ),
    "scale_pos_weight": float(
        xgb_stage2_search.best_params_["scale_pos_weight"]
    ),
}

final_xgb = xgb_stage2_search.best_estimator_
final_xgb.set_params(n_jobs=-1)

xgb_uses_class_ratio = (
    FINAL_XGB_PARAMS["scale_pos_weight"] != 1.0
)

xgb_weighting_strategy = (
    "no class reweighting (scale_pos_weight=1.0)"
)

print("\nFinal XGBoost specification frozen:")
print(f"  Training CV AUC: {FINAL_XGB_CV_AUC:.6f}")
print(f"  Parameters: {FINAL_XGB_PARAMS}")


Final XGBoost specification frozen:
  Training CV AUC: 0.737368
  Parameters: {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05, 'scale_pos_weight': 1.0}


### 2.4 Holdout evaluation and choice of explanation model (§5.1.1, Table 3)

ROC-AUC and PR-AUC on the 20% holdout. Random Forest is the explanation model because it has the
highest **training-CV** ROC-AUC (0.737385 vs 0.737368 for XGBoost); the holdout is not used for this choice.

In [25]:
# ------------------------------------------------------------
# 5. FINAL OUT-OF-SAMPLE EVALUATION ON THE 20% HOLDOUT
# ------------------------------------------------------------
# All model specifications were selected using training-only CV.
# The holdout is used only once for final out-of-sample evaluation.
#
# ROC-AUC is the primary performance metric for H1.
# PR-AUC is reported as a complementary metric because
# dissatisfaction is the minority class.
#
# No model selection or retuning is performed using holdout results.
# ------------------------------------------------------------

final_models = {
    "Logistic Regression": final_lr,
    "Random Forest": final_rf,
    "XGBoost": final_xgb,
}

model_rows = []
test_probabilities = {}

print("\n" + "=" * 72)
print("FINAL HOLDOUT PERFORMANCE")
print("=" * 72)

for name, model in final_models.items():

    probs = model.predict_proba(
        X_test
    )[:, 1]

    roc_auc = roc_auc_score(
        y_test,
        probs,
    )

    pr_auc = average_precision_score(
        y_test,
        probs,
    )

    test_probabilities[name] = probs

    model_rows.append({
        "model": name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    })

    print(
        f"{name:22s} | "
        f"ROC-AUC={roc_auc:.4f} | "
        f"PR-AUC={pr_auc:.4f}"
    )


# Holdout results are kept in the original model order.
model_comparison_df = pd.DataFrame(
    model_rows
)

model_comparison_df.to_csv(
    f"{OUT_DIR}/h1_model_comparison.csv",
    index=False,
)


# ------------------------------------------------------------
# SAVE TRAINING-CV MODEL-SELECTION INFORMATION
# ------------------------------------------------------------

selection_summary = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "final_training_cv_auc": FINAL_LR_CV_AUC,
        "final_params": str(FINAL_LR_PARAMS),
    },
    {
        "model": "Random Forest",
        "final_training_cv_auc": FINAL_RF_CV_AUC,
        "final_params": str(FINAL_RF_PARAMS),
    },
    {
        "model": "XGBoost",
        "final_training_cv_auc": FINAL_XGB_CV_AUC,
        "final_params": str(FINAL_XGB_PARAMS),
    },
])

selection_summary.to_csv(
    f"{OUT_DIR}/h1_training_cv_model_selection.csv",
    index=False,
)

print("\nTraining-CV model selection summary:")
print(
    selection_summary.to_string(
        index=False
    )
)


FINAL HOLDOUT PERFORMANCE
Logistic Regression    | ROC-AUC=0.7177 | PR-AUC=0.3769


Random Forest          | ROC-AUC=0.7505 | PR-AUC=0.4538
XGBoost                | ROC-AUC=0.7501 | PR-AUC=0.4541

Training-CV model selection summary:
              model  final_training_cv_auc                                                                                 final_params
Logistic Regression               0.713517                   {'C': 1.0, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}
      Random Forest               0.737385 {'n_estimators': 1000, 'max_depth': None, 'min_samples_leaf': 8, 'class_weight': 'balanced'}
            XGBoost               0.737368        {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05, 'scale_pos_weight': 1.0}


In [26]:
# ------------------------------------------------------------
# 6. PRIMARY EXPLANATION MODEL
# ------------------------------------------------------------
# The primary TreeSHAP model is selected using training-only
# CV ROC-AUC among the two tree ensembles.
# Holdout performance is not used for this decision.

if FINAL_RF_CV_AUC >= FINAL_XGB_CV_AUC:
    primary_explanation_model_name = "Random Forest"
    primary_explanation_model = final_rf
    primary_explanation_cv_auc = FINAL_RF_CV_AUC
else:
    primary_explanation_model_name = "XGBoost"
    primary_explanation_model = final_xgb
    primary_explanation_cv_auc = FINAL_XGB_CV_AUC

print("\nPrimary explanation model:")
print(
    f"  {primary_explanation_model_name} "
    f"(training-CV ROC-AUC={primary_explanation_cv_auc:.6f})"
)


def _positive_class_shap_values(model, X_eval):
    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(X_eval)

    # Compatibility with SHAP versions returning
    # one array per class.
    if isinstance(values, list):
        values = values[1]

    values = np.asarray(values)

    # Compatibility with SHAP versions returning
    # shape: (n_samples, n_features, n_classes)
    if values.ndim == 3:
        values = values[:, :, 1]

    if values.ndim != 2:
        raise ValueError(
            f"Unexpected SHAP output shape: {values.shape}"
        )

    return explainer, values


Primary explanation model:
  Random Forest (training-CV ROC-AUC=0.737385)


### 2.5 Global TreeSHAP importance (§5.1.2, Figure 1, Table 4)

Mean |SHAP| on the holdout, in units of predicted probability. Saves the beeswarm plot used as Figure 1.

In [27]:
# ------------------------------------------------------------
# 7. GLOBAL TREE-SHAP — PRIMARY EXPLANATION MODEL
# ------------------------------------------------------------
print("\nComputing global TreeSHAP values on the holdout sample...")

primary_explainer, primary_shap_values = (
    _positive_class_shap_values(
        primary_explanation_model,
        X_test,
    )
)

primary_shap_importance = pd.DataFrame({
    "feature": X_test.columns,
    "mean_abs_shap": np.abs(primary_shap_values).mean(axis=0),
}).sort_values(
    "mean_abs_shap",
    ascending=False,
).reset_index(drop=True)

primary_shap_importance["rank"] = (
    np.arange(1, len(primary_shap_importance) + 1)
)

primary_shap_importance["explanation_model"] = (
    primary_explanation_model_name
)

print(
    f"\n=== Global SHAP importance "
    f"({primary_explanation_model_name}): top 15 ==="
)

print(
    primary_shap_importance[
        ["rank", "feature", "mean_abs_shap"]
    ]
    .head(15)
    .to_string(index=False)
)

primary_shap_importance.to_csv(
    f"{OUT_DIR}/h1_shap_global_primary.csv",
    index=False,
)


# ------------------------------------------------------------
# SHAP SUMMARY PLOT
# ------------------------------------------------------------
plt.figure()

shap.summary_plot(
    primary_shap_values,
    X_test,
    show=False,
    max_display=15,
)

plt.tight_layout()

plt.savefig(
    f"{OUT_DIR}/phase1_shap_summary_global_primary.png",
    dpi=150,
    bbox_inches="tight",
)

plt.close()


# ------------------------------------------------------------
# FOCAL H1 PREDICTOR + PROMISED-WINDOW CONTROL
# ------------------------------------------------------------

h1_delivery_variables = [
    "delivery_delay_days",
    "estimated_delivery_days",
]

h1_delivery_shap_ranks = (
    primary_shap_importance[
        primary_shap_importance["feature"].isin(
            h1_delivery_variables
        )
    ][
        ["rank", "feature", "mean_abs_shap"]
    ]
)

print(
    "\n=== Focal delivery predictor and promised-window "
    "control in global SHAP ranking ==="
)

print(
    h1_delivery_shap_ranks.to_string(
        index=False
    )
)

h1_delivery_shap_ranks.to_csv(
    f"{OUT_DIR}/h1_shap_global_delivery_ranks.csv",
    index=False,
)


Computing global TreeSHAP values on the holdout sample...



=== Global SHAP importance (Random Forest): top 15 ===
 rank                       feature  mean_abs_shap
    1           delivery_delay_days       0.105653
    2                 freight_value       0.040352
    3                 payment_value       0.017770
    4       estimated_delivery_days       0.015308
    5                   order_price       0.014872
    6    product_description_lenght       0.014767
    7                 freight_ratio       0.013513
    8             customer_state_RJ       0.011174
    9               seller_state_SP       0.008627
   10            product_photos_qty       0.007947
   11             installment_count       0.006989
   12    category_group_electronics       0.006929
   13 category_group_furniture_home       0.006777
   14 category_group_sports_leisure       0.004148
   15                    same_state       0.003944



=== Focal delivery predictor and promised-window control in global SHAP ranking ===
 rank                 feature  mean_abs_shap
    1     delivery_delay_days       0.105653
    4 estimated_delivery_days       0.015308


### 2.6 Category-specific TreeSHAP (§5.1.4, Table 6, Appendix C)

The same frozen Random Forest specification is refitted within each category group (no re-tuning).

In [28]:
# ------------------------------------------------------------
# 8. CATEGORY-SPECIFIC TREE-SHAP
# ------------------------------------------------------------
# The globally selected primary explanation model is refit separately
# within each product-category group using the SAME frozen
# hyperparameter specification.
#
# Hyperparameters are NOT retuned within categories.
#
# Category-specific AUC is evaluated on the full category holdout.
# SHAP then evaluates whether delivery_delay_days remains the
# top-ranked predictor within each category.
#
# category_group dummy variables are removed because they have no
# within-category variation.
# ------------------------------------------------------------

print("\n" + "=" * 72)
print(
    f"CATEGORY-SPECIFIC TREE-SHAP — "
    f"{primary_explanation_model_name}"
)
print("=" * 72)

categories = sorted(
    df["category_group"]
    .dropna()
    .unique()
)

category_dummy_cols = [
    c for c in X.columns
    if c.startswith("category_group_")
]

category_shap_results = []

for cat in categories:

    # --------------------------------------------------------
    # Preserve membership from the original global 80/20 split
    # --------------------------------------------------------
    train_idx_cat = X_train.index[
        df.loc[
            X_train.index,
            "category_group",
        ] == cat
    ]

    test_idx_cat = X_test.index[
        df.loc[
            X_test.index,
            "category_group",
        ] == cat
    ]

    X_cat_train = (
        X.loc[train_idx_cat]
        .drop(
            columns=category_dummy_cols,
            errors="ignore",
        )
        .copy()
    )

    X_cat_test = (
        X.loc[test_idx_cat]
        .drop(
            columns=category_dummy_cols,
            errors="ignore",
        )
        .copy()
    )

    y_cat_train = (
        y.loc[train_idx_cat]
        .copy()
    )

    y_cat_test = (
        y.loc[test_idx_cat]
        .copy()
    )

    if (
        len(X_cat_train) == 0
        or len(X_cat_test) == 0
        or y_cat_train.nunique() < 2
        or y_cat_test.nunique() < 2
    ):
        print(
            f"{cat}: skipped because "
            f"a split lacks both classes."
        )
        continue

    # --------------------------------------------------------
    # Refit SAME frozen primary tree specification
    # within this category.
    # --------------------------------------------------------
    cat_model = clone(
        primary_explanation_model
    )

    if "n_jobs" in cat_model.get_params(
        deep=False
    ):
        cat_model.set_params(
            n_jobs=-1
        )

    cat_model.fit(
        X_cat_train,
        y_cat_train,
    )

    # Full category holdout AUC
    cat_probs = (
        cat_model.predict_proba(
            X_cat_test
        )[:, 1]
    )

    cat_auc = roc_auc_score(
        y_cat_test,
        cat_probs,
    )

    # --------------------------------------------------------
    # CATEGORY-SPECIFIC SHAP
    # --------------------------------------------------------
    _, cat_shap_values = (
        _positive_class_shap_values(
            cat_model,
            X_cat_test,
        )
    )

    cat_shap_importance = (
        pd.DataFrame({
            "feature":
                X_cat_test.columns,

            "mean_abs_shap":
                np.abs(
                    cat_shap_values
                ).mean(axis=0),
        })
        .sort_values(
            "mean_abs_shap",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    cat_shap_importance[
        "rank"
    ] = np.arange(
        1,
        len(cat_shap_importance) + 1,
    )

    cat_shap_importance[
        "category_group"
    ] = cat

    cat_shap_importance[
        "explanation_model"
    ] = primary_explanation_model_name

    cat_shap_importance.to_csv(
        f"{OUT_DIR}/h1_shap_{cat}.csv",
        index=False,
    )

    # --------------------------------------------------------
    # FOCAL H1 PREDICTOR RANK
    # --------------------------------------------------------
    delay_row = (
        cat_shap_importance[
            cat_shap_importance["feature"]
            == "delivery_delay_days"
        ]
    )

    if len(delay_row) != 1:
        raise RuntimeError(
            f"delivery_delay_days missing or duplicated "
            f"in SHAP results for {cat}."
        )

    delivery_delay_rank = int(
        delay_row["rank"].iloc[0]
    )

    delivery_delay_mean_abs_shap = float(
        delay_row[
            "mean_abs_shap"
        ].iloc[0]
    )

    # Promised-window control reported separately
    estimated_row = (
        cat_shap_importance[
            cat_shap_importance["feature"]
            == "estimated_delivery_days"
        ]
    )

    estimated_delivery_rank = int(
        estimated_row["rank"].iloc[0]
    )

    estimated_delivery_mean_abs_shap = float(
        estimated_row[
            "mean_abs_shap"
        ].iloc[0]
    )

    # --------------------------------------------------------
    # SAVE CATEGORY SUMMARY
    # --------------------------------------------------------
    category_shap_results.append({
        "category_group":
            cat,

        "n_train":
            len(y_cat_train),

        "n_test":
            len(y_cat_test),

        "n_orders_total":
            int(
                (
                    df["category_group"]
                    == cat
                ).sum()
            ),

        "n_dissatisfied_total":
            int(
                df.loc[
                    df["category_group"] == cat,
                    target,
                ].sum()
            ),

        "auc_roc":
            cat_auc,

        "top1_feature":
            cat_shap_importance.iloc[0][
                "feature"
            ],

        "top2_feature":
            cat_shap_importance.iloc[1][
                "feature"
            ],

        "top3_feature":
            cat_shap_importance.iloc[2][
                "feature"
            ],

        "delivery_delay_rank":
            delivery_delay_rank,

        "delivery_delay_mean_abs_shap":
            delivery_delay_mean_abs_shap,

        "delivery_delay_is_top1":
            delivery_delay_rank == 1,

        "estimated_delivery_rank":
            estimated_delivery_rank,

        "estimated_delivery_mean_abs_shap":
            estimated_delivery_mean_abs_shap,

        "explanation_model":
            primary_explanation_model_name,
    })

    print(
        f"{cat:20s} | "
        f"AUC={cat_auc:.4f} | "
        f"top1="
        f"{cat_shap_importance.iloc[0]['feature']} | "
        f"delivery_delay rank="
        f"{delivery_delay_rank}"
    )


# ------------------------------------------------------------
# CATEGORY SUMMARY
# ------------------------------------------------------------

category_summary_df = pd.DataFrame(
    category_shap_results
)

category_summary_df.to_csv(
    f"{OUT_DIR}/h1_category_shap_summary.csv",
    index=False,
)

print(
    "\n=== Category-specific SHAP summary ==="
)

print(
    category_summary_df.to_string(
        index=False
    )
)

n_delay_top1 = int(
    category_summary_df[
        "delivery_delay_is_top1"
    ].sum()
)

print(
    f"\ndelivery_delay_days is SHAP rank #1 in "
    f"{n_delay_top1}/"
    f"{len(category_summary_df)} categories."
)


CATEGORY-SPECIFIC TREE-SHAP — Random Forest


clothing_fashion     | AUC=0.7276 | top1=delivery_delay_days | delivery_delay rank=1


electronics          | AUC=0.6980 | top1=delivery_delay_days | delivery_delay rank=1


furniture_home       | AUC=0.7416 | top1=delivery_delay_days | delivery_delay rank=1


health_beauty        | AUC=0.7485 | top1=delivery_delay_days | delivery_delay rank=1


other                | AUC=0.7520 | top1=delivery_delay_days | delivery_delay rank=1


sports_leisure       | AUC=0.7893 | top1=delivery_delay_days | delivery_delay rank=1

=== Category-specific SHAP summary ===
  category_group  n_train  n_test  n_orders_total  n_dissatisfied_total  auc_roc        top1_feature  top2_feature               top3_feature  delivery_delay_rank  delivery_delay_mean_abs_shap  delivery_delay_is_top1  estimated_delivery_rank  estimated_delivery_mean_abs_shap explanation_model
clothing_fashion     6967    1787            8754                  1072 0.727631 delivery_delay_days freight_value                order_price                    1                      0.092993                    True                        8                          0.012837     Random Forest
     electronics    13317    3274           16591                  2273 0.697976 delivery_delay_days freight_value              payment_value                    1                      0.082176                    True                        8                          0.010405     Random 

### 2.7 Drop-column ablation — primary H1 test (§5.1.3, Table 5, Appendix A)

Each predictor (or the full block of one-hot columns of a categorical variable) is removed, the
Random Forest is refitted, and the loss in holdout ROC-AUC is recorded. Removing the promised-date
deviation costs 0.113, about 19 times the next-largest loss.

In [29]:
# ------------------------------------------------------------
# 9. PRIMARY H1 CONFIRMATION:
#    DROP-COLUMN ABLATION ACROSS CONCEPTUAL PREDICTORS
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("H1 PRIMARY TEST — DROP-COLUMN PREDICTOR ABLATION")
print("=" * 72)

# Each conceptual predictor is removed separately.
# Categorical predictors are treated as one conceptual variable:
# all corresponding dummy columns are removed together.

predictor_blocks = {
    # Focal H1 predictor
    "delivery_delay_days": [
        "delivery_delay_days",
    ],

    # Promised-window / expectation control
    "estimated_delivery_days": [
        "estimated_delivery_days",
    ],

    # Commercial characteristics
    "order_price": [
        "order_price",
    ],
    "freight_value": [
        "freight_value",
    ],
    "freight_ratio": [
        "freight_ratio",
    ],

    # Payment characteristics
    "installment_count": [
        "installment_count",
    ],
    "payment_value": [
        "payment_value",
    ],
    "payment_type": [
        c for c in X.columns
        if c.startswith("payment_type_")
    ],

    # Product characteristics
    "product_description_lenght": [
        "product_description_lenght",
    ],
    "product_photos_qty": [
        "product_photos_qty",
    ],
    "has_missing_product_info": [
        "has_missing_product_info",
    ],

    # Geography
    "same_state": [
        "same_state",
    ],
    "customer_state": [
        c for c in X.columns
        if c.startswith("customer_state_")
    ],
    "seller_state": [
        c for c in X.columns
        if c.startswith("seller_state_")
    ],

    # Product category
    "category_group": [
        c for c in X.columns
        if c.startswith("category_group_")
    ],
}

# Keep only columns actually present in X.
predictor_blocks = {
    predictor: [
        c for c in cols
        if c in X.columns
    ]
    for predictor, cols in predictor_blocks.items()
}

print("\nPredictor blocks:")
for predictor, cols in predictor_blocks.items():
    print(
        f"  {predictor:30s}: "
        f"{len(cols)} column(s)"
    )


# ------------------------------------------------------------
# FULL PRIMARY EXPLANATION MODEL PERFORMANCE
# ------------------------------------------------------------
# Use the tree ensemble selected by training-only CV.
# In the current run this is Random Forest, but we do not hardcode it.

full_probs_primary = (
    primary_explanation_model.predict_proba(
        X_test
    )[:, 1]
)

full_auc_primary = roc_auc_score(
    y_test,
    full_probs_primary,
)

print(
    f"\nPrimary explanation model: "
    f"{primary_explanation_model_name}"
)

print(
    f"Full holdout ROC-AUC: "
    f"{full_auc_primary:.6f}"
)


# ------------------------------------------------------------
# DROP-COLUMN ABLATION
# ------------------------------------------------------------

ablation_rows = []

for predictor, cols_to_drop in predictor_blocks.items():

    if len(cols_to_drop) == 0:
        continue

    X_train_reduced = X_train.drop(
        columns=cols_to_drop
    )

    X_test_reduced = X_test.drop(
        columns=cols_to_drop
    )

    # Refit the SAME frozen model specification after removing
    # only the focal conceptual predictor.
    reduced_model = clone(
        primary_explanation_model
    )

    reduced_model.fit(
        X_train_reduced,
        y_train,
    )

    reduced_probs = (
        reduced_model.predict_proba(
            X_test_reduced
        )[:, 1]
    )

    reduced_auc = roc_auc_score(
        y_test,
        reduced_probs,
    )

    auc_drop = (
        full_auc_primary
        - reduced_auc
    )

    ablation_rows.append({
        "predictor": predictor,
        "n_columns_removed": len(cols_to_drop),
        "full_auc": full_auc_primary,
        "ablated_auc": reduced_auc,
        "auc_drop": auc_drop,
        "explanation_model": primary_explanation_model_name,
    })

    print(
        f"{predictor:30s} | "
        f"removed={len(cols_to_drop):2d} | "
        f"AUC={reduced_auc:.4f} | "
        f"drop={auc_drop:.4f}"
    )


# ------------------------------------------------------------
# RANK PREDICTORS BY LOSS IN OUT-OF-SAMPLE AUC
# ------------------------------------------------------------

ablation_df = (
    pd.DataFrame(ablation_rows)
    .sort_values(
        "auc_drop",
        ascending=False,
    )
    .reset_index(drop=True)
)

ablation_df[
    "rank_by_auc_drop"
] = np.arange(
    1,
    len(ablation_df) + 1,
)

print(
    "\n=== Global predictor-ablation ranking ==="
)

print(
    ablation_df.to_string(
        index=False
    )
)

ablation_df.to_csv(
    f"{OUT_DIR}/h1_predictor_ablation.csv",
    index=False,
)


# ------------------------------------------------------------
# FOCAL H1 RESULT
# ------------------------------------------------------------

delay_ablation_row = (
    ablation_df[
        ablation_df["predictor"]
        == "delivery_delay_days"
    ]
)

if len(delay_ablation_row) != 1:
    raise RuntimeError(
        "delivery_delay_days missing or duplicated "
        "in global ablation results."
    )

delay_ablation_rank = int(
    delay_ablation_row[
        "rank_by_auc_drop"
    ].iloc[0]
)

delay_ablation_drop = float(
    delay_ablation_row[
        "auc_drop"
    ].iloc[0]
)

delay_largest_auc_drop = (
    delay_ablation_rank == 1
)

print(
    f"\nGlobal delivery_delay_days "
    f"ablation rank = {delay_ablation_rank}"
)

print(
    f"Global delivery_delay_days "
    f"AUC drop = {delay_ablation_drop:.6f}"
)

print(
    f"delivery_delay_days has largest "
    f"AUC drop: {delay_largest_auc_drop}"
)


H1 PRIMARY TEST — DROP-COLUMN PREDICTOR ABLATION

Predictor blocks:
  delivery_delay_days           : 1 column(s)
  estimated_delivery_days       : 1 column(s)
  order_price                   : 1 column(s)
  freight_value                 : 1 column(s)
  freight_ratio                 : 1 column(s)
  installment_count             : 1 column(s)
  payment_value                 : 1 column(s)
  payment_type                  : 4 column(s)
  product_description_lenght    : 1 column(s)
  product_photos_qty            : 1 column(s)
  has_missing_product_info      : 1 column(s)
  same_state                    : 1 column(s)
  customer_state                : 26 column(s)
  seller_state                  : 21 column(s)
  category_group                : 5 column(s)



Primary explanation model: Random Forest
Full holdout ROC-AUC: 0.750495


delivery_delay_days            | removed= 1 | AUC=0.6372 | drop=0.1133


estimated_delivery_days        | removed= 1 | AUC=0.7463 | drop=0.0042


order_price                    | removed= 1 | AUC=0.7496 | drop=0.0009


freight_value                  | removed= 1 | AUC=0.7445 | drop=0.0060


... (7 outputs left out)

same_state                     | removed= 1 | AUC=0.7502 | drop=0.0003


customer_state                 | removed=26 | AUC=0.7518 | drop=-0.0013


seller_state                   | removed=21 | AUC=0.7477 | drop=0.0028


category_group                 | removed= 5 | AUC=0.7447 | drop=0.0058

=== Global predictor-ablation ranking ===
                 predictor  n_columns_removed  full_auc  ablated_auc  auc_drop explanation_model  rank_by_auc_drop
       delivery_delay_days                  1  0.750495     0.637207  0.113288     Random Forest                 1
             freight_value                  1  0.750495     0.744545  0.005950     Random Forest                 2
            category_group                  5  0.750495     0.744701  0.005795     Random Forest                 3
product_description_lenght                  1  0.750495     0.745852  0.004643     Random Forest                 4
   estimated_delivery_days                  1  0.750495     0.746324  0.004171     Random Forest                 5
        product_photos_qty                  1  0.750495     0.747525  0.002970     Random Forest                 6
              seller_state                 21  0.750495     0.747659  0.002837   

### 2.8 Category-specific ablation (§5.1.4, Table 6, Appendix C)

In [30]:
# ------------------------------------------------------------
# 9B. CATEGORY-SPECIFIC PREDICTOR ABLATION
#
# Tests whether delivery_delay_days remains the strongest
# predictive contributor within each product-category group.
#
# The SAME frozen primary tree specification is used in every
# category. Hyperparameters are not retuned by category.
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("H1 — CATEGORY-SPECIFIC PREDICTOR ABLATION")
print("=" * 72)

categories = sorted(
    df["category_group"]
    .dropna()
    .unique()
)

# category_group dummies are constant within a given category,
# so they are excluded from within-category models.
category_dummy_cols = [
    c for c in X.columns
    if c.startswith("category_group_")
]

category_ablation_rows = []

for cat in categories:

    # Preserve membership from the original global 80/20 split
    train_idx_cat = X_train.index[
        df.loc[
            X_train.index,
            "category_group"
        ] == cat
    ]

    test_idx_cat = X_test.index[
        df.loc[
            X_test.index,
            "category_group"
        ] == cat
    ]

    X_cat_train = (
        X.loc[train_idx_cat]
        .drop(
            columns=category_dummy_cols,
            errors="ignore",
        )
        .copy()
    )

    X_cat_test = (
        X.loc[test_idx_cat]
        .drop(
            columns=category_dummy_cols,
            errors="ignore",
        )
        .copy()
    )

    y_cat_train = y.loc[
        train_idx_cat
    ].copy()

    y_cat_test = y.loc[
        test_idx_cat
    ].copy()

    if (
        len(X_cat_train) == 0
        or len(X_cat_test) == 0
        or y_cat_train.nunique() < 2
        or y_cat_test.nunique() < 2
    ):
        print(
            f"{cat}: skipped because "
            f"a split lacks both classes."
        )
        continue

    # --------------------------------------------------------
    # Full model within category
    # --------------------------------------------------------
    cat_full_model = clone(
        primary_explanation_model
    )

    if "n_jobs" in cat_full_model.get_params(
        deep=False
    ):
        cat_full_model.set_params(
            n_jobs=-1
        )

    cat_full_model.fit(
        X_cat_train,
        y_cat_train,
    )

    cat_full_probs = (
        cat_full_model.predict_proba(
            X_cat_test
        )[:, 1]
    )

    cat_full_auc = roc_auc_score(
        y_cat_test,
        cat_full_probs,
    )

    cat_temp_rows = []

    # --------------------------------------------------------
    # Remove each conceptual predictor separately
    # --------------------------------------------------------
    for predictor, cols_to_drop in predictor_blocks.items():

        # Product category itself is constant within category,
        # so it cannot be evaluated here.
        if predictor == "category_group":
            continue

        valid_cols = [
            c for c in cols_to_drop
            if c in X_cat_train.columns
        ]

        if len(valid_cols) == 0:
            continue

        reduced_model = clone(
            primary_explanation_model
        )

        if "n_jobs" in reduced_model.get_params(
            deep=False
        ):
            reduced_model.set_params(
                n_jobs=-1
            )

        reduced_model.fit(
            X_cat_train.drop(
                columns=valid_cols
            ),
            y_cat_train,
        )

        reduced_probs = (
            reduced_model.predict_proba(
                X_cat_test.drop(
                    columns=valid_cols
                )
            )[:, 1]
        )

        reduced_auc = roc_auc_score(
            y_cat_test,
            reduced_probs,
        )

        auc_drop = (
            cat_full_auc
            - reduced_auc
        )

        cat_temp_rows.append({
            "category_group": cat,
            "predictor": predictor,
            "n_columns_removed": len(valid_cols),
            "full_auc": cat_full_auc,
            "ablated_auc": reduced_auc,
            "auc_drop": auc_drop,
            "explanation_model":
                primary_explanation_model_name,
        })

    # --------------------------------------------------------
    # Rank predictors within category
    # --------------------------------------------------------
    cat_temp_df = (
        pd.DataFrame(cat_temp_rows)
        .sort_values(
            "auc_drop",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    cat_temp_df[
        "rank_by_auc_drop"
    ] = np.arange(
        1,
        len(cat_temp_df) + 1,
    )

    category_ablation_rows.extend(
        cat_temp_df.to_dict(
            "records"
        )
    )

    delay_row_cat = (
        cat_temp_df[
            cat_temp_df["predictor"]
            == "delivery_delay_days"
        ]
    )

    if len(delay_row_cat) != 1:
        raise RuntimeError(
            f"delivery_delay_days missing or duplicated "
            f"in ablation results for {cat}."
        )

    delay_rank_cat = int(
        delay_row_cat[
            "rank_by_auc_drop"
        ].iloc[0]
    )

    delay_drop_cat = float(
        delay_row_cat[
            "auc_drop"
        ].iloc[0]
    )

    print(
        f"{cat:20s} | "
        f"full AUC={cat_full_auc:.4f} | "
        f"delay rank={delay_rank_cat} | "
        f"delay drop={delay_drop_cat:.4f}"
    )


# ------------------------------------------------------------
# SAVE ALL CATEGORY × PREDICTOR ABLATION RESULTS
# ------------------------------------------------------------

category_ablation_df = pd.DataFrame(
    category_ablation_rows
)

category_ablation_df.to_csv(
    f"{OUT_DIR}/h1_category_predictor_ablation.csv",
    index=False,
)


# ------------------------------------------------------------
# COMPACT SUMMARY FOR THE FOCAL H1 PREDICTOR
# ------------------------------------------------------------

category_delay_summary = (
    category_ablation_df[
        category_ablation_df[
            "predictor"
        ] == "delivery_delay_days"
    ][
        [
            "category_group",
            "full_auc",
            "ablated_auc",
            "auc_drop",
            "rank_by_auc_drop",
        ]
    ]
    .sort_values(
        "category_group"
    )
    .reset_index(drop=True)
)

category_delay_summary[
    "delivery_delay_is_strongest"
] = (
    category_delay_summary[
        "rank_by_auc_drop"
    ] == 1
)

category_delay_summary.to_csv(
    f"{OUT_DIR}/h1_category_delivery_delay_ablation_summary.csv",
    index=False,
)

print(
    "\n=== Category delivery-delay ablation summary ==="
)

print(
    category_delay_summary.to_string(
        index=False
    )
)

n_delay_top1_ablation = int(
    category_delay_summary[
        "delivery_delay_is_strongest"
    ].sum()
)

print(
    f"\ndelivery_delay_days has the largest AUC drop in "
    f"{n_delay_top1_ablation}/"
    f"{len(category_delay_summary)} categories."
)


H1 — CATEGORY-SPECIFIC PREDICTOR ABLATION


clothing_fashion     | full AUC=0.7276 | delay rank=1 | delay drop=0.1075


electronics          | full AUC=0.6980 | delay rank=1 | delay drop=0.0936


furniture_home       | full AUC=0.7416 | delay rank=1 | delay drop=0.0968


health_beauty        | full AUC=0.7485 | delay rank=1 | delay drop=0.1336


other                | full AUC=0.7520 | delay rank=1 | delay drop=0.1306


sports_leisure       | full AUC=0.7893 | delay rank=1 | delay drop=0.1209

=== Category delivery-delay ablation summary ===
  category_group  full_auc  ablated_auc  auc_drop  rank_by_auc_drop  delivery_delay_is_strongest
clothing_fashion  0.727631     0.620083  0.107548                 1                         True
     electronics  0.697976     0.604386  0.093589                 1                         True
  furniture_home  0.741629     0.644875  0.096754                 1                         True
   health_beauty  0.748483     0.614899  0.133584                 1                         True
           other  0.751974     0.621394  0.130580                 1                         True
  sports_leisure  0.789256     0.668335  0.120922                 1                         True

delivery_delay_days has the largest AUC drop in 6/6 categories.


### 2.9 Robustness (§5.4, Appendix G.2–G.5)

- **A. Seller-grouped five-fold CV** (G.2): all orders of a seller stay in the same fold.
- **B. Repeated random splits** at 70/30, 80/20 and 90/10 with five seeds (G.3).
- **C. Chronological holdout** (G.4): train on the earliest 80%, test on the latest 20%, and repeat
  the ablation. This cell also computes the sample-selection bounds for orders without a delivery date.
- **D. Shared reviews** (G.5): repeat the ablation without the 1,008 orders whose review is shared.

In [31]:
# ------------------------------------------------------------
# 10. ROBUSTNESS A: SELLER-GROUPED CV
#
# Tests whether predictive performance persists when orders
# associated with the same seller are kept within the same fold.
#
# Model specifications remain frozen; no tuning is performed here.
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("H1 ROBUSTNESS A — SELLER-GROUPED CV")
print("=" * 72)

# Align seller groups explicitly with the modelling sample
seller_groups_cv = (
    seller_groups
    .loc[X.index]
    .astype("object")
    .copy()
)

n_missing_sellers = int(
    seller_groups_cv.isna().sum()
)

# If seller_id is missing, do not treat unrelated missing observations
# as belonging to one common seller. Give each its own pseudo-group.
if n_missing_sellers > 0:

    missing_idx = seller_groups_cv.index[
        seller_groups_cv.isna()
    ]

    seller_groups_cv.loc[missing_idx] = [
        f"__missing_seller_{i}__"
        for i in range(len(missing_idx))
    ]

print(
    f"Missing seller IDs: "
    f"{n_missing_sellers}"
)


# ------------------------------------------------------------
# SELLER STRUCTURE
# ------------------------------------------------------------

seller_counts = (
    seller_groups_cv
    .value_counts()
)

seller_structure_df = pd.DataFrame([{
    "n_unique_sellers":
        seller_groups_cv.nunique(),

    "n_missing_seller_ids":
        n_missing_sellers,

    "orders_per_seller_mean":
        seller_counts.mean(),

    "orders_per_seller_median":
        seller_counts.median(),

    "orders_per_seller_p90":
        seller_counts.quantile(0.90),

    "orders_per_seller_p95":
        seller_counts.quantile(0.95),

    "orders_per_seller_max":
        seller_counts.max(),
}])

print("\nSeller structure:")
print(
    seller_structure_df.to_string(
        index=False
    )
)

seller_structure_df.to_csv(
    f"{OUT_DIR}/h1_seller_group_structure.csv",
    index=False,
)


# ------------------------------------------------------------
# 5-FOLD STRATIFIED GROUP CV
# ------------------------------------------------------------

seller_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

seller_cv_rows = []

for name, model in final_models.items():

    m = clone(model)

    # Avoid nested parallelism:
    # outer CV runs folds in parallel.
    if "n_jobs" in m.get_params(
        deep=False
    ):
        m.set_params(
            n_jobs=1
        )

    scores = cross_val_score(
        m,
        X,
        y,
        groups=seller_groups_cv,
        cv=seller_cv,
        scoring="roc_auc",
        n_jobs=-1,
    )

    seller_cv_rows.append({
        "model": name,
        "seller_grouped_cv_auc_mean":
            scores.mean(),
        "seller_grouped_cv_auc_std":
            scores.std(),
    })

    print(
        f"{name:22s}: "
        f"{scores.mean():.4f} "
        f"+/- {scores.std():.4f}"
    )


seller_cv_df = pd.DataFrame(
    seller_cv_rows
)

seller_cv_df.to_csv(
    f"{OUT_DIR}/h1_seller_grouped_cv_auc.csv",
    index=False,
)


H1 ROBUSTNESS A — SELLER-GROUPED CV
Missing seller IDs: 0

Seller structure:
 n_unique_sellers  n_missing_seller_ids  orders_per_seller_mean  orders_per_seller_median  orders_per_seller_p90  orders_per_seller_p95  orders_per_seller_max
             2956                     0               32.418809                       7.0                   70.5                 128.25                   1795


Logistic Regression   : 0.7124 +/- 0.0091


Random Forest         : 0.7242 +/- 0.0138


XGBoost               : 0.7282 +/- 0.0140


In [32]:
# ------------------------------------------------------------
# 11. ROBUSTNESS B:
#     ALTERNATIVE STRATIFIED RANDOM SPLIT RATIOS
# ------------------------------------------------------------
# Tests whether out-of-sample performance is stable across
# alternative train/test proportions and random partitions.
#
# Model specifications remain frozen. No tuning is performed.
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("H1 ROBUSTNESS B — SPLIT-RATIO SENSITIVITY")
print("=" * 72)

test_fractions = [0.10, 0.20, 0.30]

# Fixed, prespecified seeds used only for robustness assessment.
split_seeds = [42, 123, 2026, 7, 99]

split_rows = []

for test_fraction in test_fractions:

    for seed in split_seeds:

        Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(
            X,
            y,
            test_size=test_fraction,
            stratify=y,
            random_state=seed,
        )

        row = {
            "split":
                f"{int((1-test_fraction)*100)}/"
                f"{int(test_fraction*100)}",
            "test_fraction": test_fraction,
            "random_seed": seed,
            "n_train": len(ytr_s),
            "n_test": len(yte_s),
        }

        for name, model in final_models.items():

            m = clone(model)

            # Fits are sequential here, so allow each model
            # to use available cores where supported.
            if "n_jobs" in m.get_params(deep=False):
                m.set_params(n_jobs=-1)

            m.fit(
                Xtr_s,
                ytr_s,
            )

            probs = m.predict_proba(
                Xte_s
            )[:, 1]

            row[f"{name}_auc"] = roc_auc_score(
                yte_s,
                probs,
            )

        split_rows.append(row)


# ------------------------------------------------------------
# SAVE ALL INDIVIDUAL RUNS
# ------------------------------------------------------------

split_sensitivity_runs_df = pd.DataFrame(
    split_rows
)

split_sensitivity_runs_df.to_csv(
    f"{OUT_DIR}/h1_split_ratio_sensitivity_all_runs.csv",
    index=False,
)


# ------------------------------------------------------------
# SUMMARISE ACROSS RANDOM PARTITIONS
# ------------------------------------------------------------

auc_cols = [
    f"{name}_auc"
    for name in final_models.keys()
]

summary_rows = []

for split_name, group in split_sensitivity_runs_df.groupby(
    "split",
    sort=False,
):

    row = {
        "split": split_name,
        "n_repetitions": len(group),
        "n_train": int(group["n_train"].iloc[0]),
        "n_test": int(group["n_test"].iloc[0]),
    }

    for col in auc_cols:
        model_name = col.replace("_auc", "")

        row[f"{model_name}_auc_mean"] = (
            group[col].mean()
        )

        row[f"{model_name}_auc_std"] = (
            group[col].std()
        )

        row[f"{model_name}_auc_min"] = (
            group[col].min()
        )

        row[f"{model_name}_auc_max"] = (
            group[col].max()
        )

    summary_rows.append(row)


split_sensitivity_df = pd.DataFrame(
    summary_rows
)

print("\n=== Split-ratio robustness summary ===")
print(
    split_sensitivity_df.to_string(
        index=False
    )
)

split_sensitivity_df.to_csv(
    f"{OUT_DIR}/h1_split_ratio_sensitivity.csv",
    index=False,
)


H1 ROBUSTNESS B — SPLIT-RATIO SENSITIVITY



=== Split-ratio robustness summary ===
split  n_repetitions  n_train  n_test  Logistic Regression_auc_mean  Logistic Regression_auc_std  Logistic Regression_auc_min  Logistic Regression_auc_max  Random Forest_auc_mean  Random Forest_auc_std  Random Forest_auc_min  Random Forest_auc_max  XGBoost_auc_mean  XGBoost_auc_std  XGBoost_auc_min  XGBoost_auc_max
90/10              5    86247    9583                      0.722430                     0.005350                     0.714452                     0.727616                0.748949               0.004709               0.741544               0.753436          0.751133         0.004083         0.746747         0.756963
80/20              5    76664   19166                      0.719626                     0.004583                     0.714444                     0.725885                0.748029               0.002060               0.744944               0.750495          0.749557         0.002645         0.746456         0.752332
70/30    

In [33]:
# ------------------------------------------------------------
# 12. ROBUSTNESS C: TEMPORAL HOLDOUT
#
# Models are trained on the earliest 80% of orders and evaluated
# on the latest 20%, preserving chronological ordering.
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("H1 ROBUSTNESS C — TEMPORAL HOLDOUT")
print("=" * 72)

temporal_cutoff = (
    df[
        "order_purchase_timestamp"
    ]
    .quantile(0.80)
)

temporal_train_mask = (
    df[
        "order_purchase_timestamp"
    ] <= temporal_cutoff
)

temporal_test_mask = (
    df[
        "order_purchase_timestamp"
    ] > temporal_cutoff
)

X_train_t = X.loc[
    temporal_train_mask
].copy()

X_test_t = X.loc[
    temporal_test_mask
].copy()

y_train_t = y.loc[
    temporal_train_mask
].copy()

y_test_t = y.loc[
    temporal_test_mask
].copy()

print(
    f"Temporal cutoff: "
    f"{temporal_cutoff}"
)

print(
    f"  Train n={len(y_train_t):,}, "
    f"dissatisfied="
    f"{y_train_t.mean():.4f}"
)

print(
    f"  Test  n={len(y_test_t):,}, "
    f"dissatisfied="
    f"{y_test_t.mean():.4f}"
)

temporal_rows = []

for name, model in final_models.items():

    m = clone(model)

    m.fit(
        X_train_t,
        y_train_t,
    )

    probs = (
        m.predict_proba(
            X_test_t
        )[:, 1]
    )

    temporal_rows.append({
        "model":
            name,

        "split":
            "temporal_80_20",

        "cutoff":
            str(temporal_cutoff),

        "train_n":
            len(y_train_t),

        "test_n":
            len(y_test_t),

        "train_dissatisfaction_rate":
            y_train_t.mean(),

        "test_dissatisfaction_rate":
            y_test_t.mean(),

        "roc_auc":
            roc_auc_score(
                y_test_t,
                probs,
            ),

        "pr_auc":
            average_precision_score(
                y_test_t,
                probs,
            ),
    })


temporal_df = pd.DataFrame(
    temporal_rows
)

print(
    temporal_df[
        [
            "model",
            "roc_auc",
            "pr_auc",
        ]
    ].to_string(
        index=False
    )
)

temporal_df.to_csv(
    f"{OUT_DIR}/h1_temporal_model_comparison.csv",
    index=False,
)
# ------------------------------------------------------------
# 12B. ROBUSTNESS C2:
#      TEMPORAL PREDICTOR ABLATION
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("H1 ROBUSTNESS C2 — TEMPORAL PREDICTOR ABLATION")
print("=" * 72)

# Full frozen primary explanation-model specification
temporal_full_model = clone(
    primary_explanation_model
)

if "n_jobs" in temporal_full_model.get_params(deep=False):
    temporal_full_model.set_params(n_jobs=-1)

temporal_full_model.fit(
    X_train_t,
    y_train_t,
)

temporal_full_probs = (
    temporal_full_model.predict_proba(
        X_test_t
    )[:, 1]
)

temporal_full_auc = roc_auc_score(
    y_test_t,
    temporal_full_probs,
)

temporal_ablation_rows = []

for predictor, cols_to_drop in predictor_blocks.items():

    valid_cols = [
        c for c in cols_to_drop
        if c in X_train_t.columns
    ]

    if len(valid_cols) == 0:
        continue

    reduced_model = clone(
        primary_explanation_model
    )

    if "n_jobs" in reduced_model.get_params(deep=False):
        reduced_model.set_params(n_jobs=-1)

    reduced_model.fit(
        X_train_t.drop(columns=valid_cols),
        y_train_t,
    )

    reduced_probs = (
        reduced_model.predict_proba(
            X_test_t.drop(columns=valid_cols)
        )[:, 1]
    )

    reduced_auc = roc_auc_score(
        y_test_t,
        reduced_probs,
    )

    temporal_ablation_rows.append({
        "predictor": predictor,
        "full_temporal_auc": temporal_full_auc,
        "ablated_temporal_auc": reduced_auc,
        "auc_drop": temporal_full_auc - reduced_auc,
        "explanation_model": primary_explanation_model_name,
    })


temporal_ablation_df = (
    pd.DataFrame(temporal_ablation_rows)
    .sort_values(
        "auc_drop",
        ascending=False,
    )
    .reset_index(drop=True)
)

temporal_ablation_df[
    "rank_by_auc_drop"
] = np.arange(
    1,
    len(temporal_ablation_df) + 1,
)

print(
    "\n=== Temporal predictor-ablation ranking ==="
)

print(
    temporal_ablation_df.to_string(
        index=False
    )
)

temporal_ablation_df.to_csv(
    f"{OUT_DIR}/h1_temporal_predictor_ablation.csv",
    index=False,
)

temporal_delay_row = (
    temporal_ablation_df[
        temporal_ablation_df["predictor"]
        == "delivery_delay_days"
    ]
)

temporal_delay_rank = int(
    temporal_delay_row[
        "rank_by_auc_drop"
    ].iloc[0]
)

temporal_delay_auc_drop = float(
    temporal_delay_row[
        "auc_drop"
    ].iloc[0]
)

temporal_delay_is_strongest = (
    temporal_delay_rank == 1
)

print(
    f"\nTemporal delivery_delay_days rank: "
    f"{temporal_delay_rank}"
)

print(
    f"Temporal delivery_delay_days AUC drop: "
    f"{temporal_delay_auc_drop:.6f}"
)

# ------------------------------------------------------------
# 13. SAMPLE-SELECTION SENSITIVITY:
#     MISSING DELIVERY-DATE BOUNDS
#
# This is not a model-performance test.
# It documents the possible dissatisfaction prevalence among
# orders excluded because realized delivery timing is unavailable.
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("H1 SAMPLE SENSITIVITY — MISSING DELIVERY-DATE BOUNDS")
print("=" * 72)

orders_bounds = pd.read_csv(
    f"{DATA_DIR}/olist_orders_dataset.csv"
)

reviews_bounds = pd.read_csv(
    f"{DATA_DIR}/olist_order_reviews_dataset.csv"
)

reviews_bounds[
    "review_creation_date"
] = pd.to_datetime(
    reviews_bounds[
        "review_creation_date"
    ]
)

reviews_bounds_latest = (
    reviews_bounds
    .sort_values(
        "review_creation_date"
    )
    .drop_duplicates(
        subset="order_id",
        keep="last",
    )[
        [
            "order_id",
            "review_score",
        ]
    ]
)

bounds_raw = (
    orders_bounds
    .merge(
        reviews_bounds_latest,
        on="order_id",
        how="left",
    )
)

missing_delivery_observed_review = (
    bounds_raw[
        bounds_raw[
            "order_delivered_customer_date"
        ].isna()
        &
        bounds_raw[
            "review_score"
        ].notna()
    ]
)

n_missing_delivery_observed_review = (
    len(
        missing_delivery_observed_review
    )
)

n_observed_sample = len(df)

n_observed_dissatisfied = int(
    y.sum()
)

n_bounds_total = (
    n_observed_sample
    + n_missing_delivery_observed_review
)

lower_dissatisfied = (
    n_observed_dissatisfied
)

upper_dissatisfied = (
    n_observed_dissatisfied
    + n_missing_delivery_observed_review
)

observed_rate = (
    n_observed_dissatisfied
    / n_observed_sample
)

lower_rate = (
    lower_dissatisfied
    / n_bounds_total
)

upper_rate = (
    upper_dissatisfied
    / n_bounds_total
)


bounds_df = pd.DataFrame([
    {
        "scenario":
            "Observed analytical sample",

        "n_orders":
            n_observed_sample,

        "n_dissatisfied":
            n_observed_dissatisfied,

        "dissatisfaction_rate":
            observed_rate,
    },

    {
        "scenario":
            "Lower bound: excluded orders all satisfied",

        "n_orders":
            n_bounds_total,

        "n_dissatisfied":
            lower_dissatisfied,

        "dissatisfaction_rate":
            lower_rate,
    },

    {
        "scenario":
            "Upper bound: excluded orders all dissatisfied",

        "n_orders":
            n_bounds_total,

        "n_dissatisfied":
            upper_dissatisfied,

        "dissatisfaction_rate":
            upper_rate,
    },
])

print(
    f"Missing delivery date with "
    f"observed review: "
    f"{n_missing_delivery_observed_review:,}"
)

print(
    bounds_df.to_string(
        index=False
    )
)

bounds_df.to_csv(
    f"{OUT_DIR}/h1_missing_delivery_date_bounds.csv",
    index=False,
)


H1 ROBUSTNESS C — TEMPORAL HOLDOUT
Temporal cutoff: 2018-05-27 11:08:40
  Train n=76,664, dissatisfied=0.1359
  Test  n=19,166, dissatisfied=0.0968


              model  roc_auc   pr_auc
Logistic Regression 0.638786 0.200284
      Random Forest 0.655980 0.266922
            XGBoost 0.663004 0.270937

H1 ROBUSTNESS C2 — TEMPORAL PREDICTOR ABLATION



=== Temporal predictor-ablation ranking ===
                 predictor  full_temporal_auc  ablated_temporal_auc  auc_drop explanation_model  rank_by_auc_drop
       delivery_delay_days            0.65598              0.572484  0.083496     Random Forest                 1
            category_group            0.65598              0.650701  0.005279     Random Forest                 2
product_description_lenght            0.65598              0.651889  0.004092     Random Forest                 3
        product_photos_qty            0.65598              0.652750  0.003230     Random Forest                 4
             freight_value            0.65598              0.652783  0.003197     Random Forest                 5
   estimated_delivery_days            0.65598              0.653892  0.002088     Random Forest                 6
             payment_value            0.65598              0.655026  0.000954     Random Forest                 7
              seller_state            0.655

Missing delivery date with observed review: 2,843
                                     scenario  n_orders  n_dissatisfied  dissatisfaction_rate
                   Observed analytical sample     95830           12276              0.128102
   Lower bound: excluded orders all satisfied     98673           12276              0.124411
Upper bound: excluded orders all dissatisfied     98673           15119              0.153223


In [34]:
# ------------------------------------------------------------
# ROBUSTNESS D: H1 ablation without the 1,008 shared-review orders (Appendix G.5)
# ------------------------------------------------------------
rl = (order_reviews.sort_values("review_creation_date")
      .drop_duplicates("order_id", keep="last")[["order_id", "review_id"]])
shared_order_ids = set(rl.loc[rl["review_id"].duplicated(keep=False), "order_id"])
print(f"Shared-review orders in the H1 sample: {df['order_id'].isin(shared_order_ids).sum():,}")

train_keep = X_train.index[~df.loc[X_train.index, "order_id"].isin(shared_order_ids)]
test_keep  = X_test.index[~df.loc[X_test.index, "order_id"].isin(shared_order_ids)]

X_train_excl, y_train_excl = X_train.loc[train_keep], y_train.loc[train_keep]
X_test_excl,  y_test_excl  = X_test.loc[test_keep],  y_test.loc[test_keep]

print(f"Training sample: {len(X_train):,} -> {len(X_train_excl):,}")
print(f"Holdout sample:  {len(X_test):,} -> {len(X_test_excl):,}")

# Refit the SAME frozen primary explanation model (no retuning)
excl_model = clone(primary_explanation_model)
excl_model.fit(X_train_excl, y_train_excl)

full_probs_excl = excl_model.predict_proba(X_test_excl)[:, 1]
full_auc_excl = roc_auc_score(y_test_excl, full_probs_excl)
print(f"\n{primary_explanation_model_name} full holdout ROC-AUC (excl. shared-review orders): {full_auc_excl:.6f}")
print(f"(Compare with the primary-sample value: {full_auc_primary:.6f})")

# Re-run the drop-column ablation on the reduced sample
excl_ablation_rows = []
for predictor, cols_to_drop in predictor_blocks.items():
    if len(cols_to_drop) == 0:
        continue
    reduced_model = clone(primary_explanation_model)
    reduced_model.fit(X_train_excl.drop(columns=cols_to_drop), y_train_excl)
    reduced_probs = reduced_model.predict_proba(X_test_excl.drop(columns=cols_to_drop))[:, 1]
    reduced_auc = roc_auc_score(y_test_excl, reduced_probs)
    excl_ablation_rows.append({
        "predictor": predictor,
        "ablated_auc": reduced_auc,
        "auc_drop": full_auc_excl - reduced_auc,
    })

excl_ablation_df = (pd.DataFrame(excl_ablation_rows)
                     .sort_values("auc_drop", ascending=False)
                     .reset_index(drop=True))
excl_ablation_df["rank_by_auc_drop"] = np.arange(1, len(excl_ablation_df) + 1)
print("\n=== Predictor-ablation ranking, excluding shared-review orders ===")
print(excl_ablation_df.to_string(index=False))

excl_ablation_df.to_csv(f"{OUT_DIR}/h1_ablation_without_shared_reviews.csv", index=False)

Shared-review orders in the H1 sample: 1,008
Training sample: 76,664 -> 75,857
Holdout sample:  19,166 -> 18,965



Random Forest full holdout ROC-AUC (excl. shared-review orders): 0.750881
(Compare with the primary-sample value: 0.750495)



=== Predictor-ablation ranking, excluding shared-review orders ===
                 predictor  ablated_auc  auc_drop  rank_by_auc_drop
       delivery_delay_days     0.636279  0.114602                 1
            category_group     0.743207  0.007675                 2
             freight_value     0.743827  0.007055                 3
product_description_lenght     0.746501  0.004381                 4
   estimated_delivery_days     0.746658  0.004223                 5
        product_photos_qty     0.747956  0.002925                 6
              seller_state     0.748846  0.002036                 7
             payment_value     0.749752  0.001130                 8
                same_state     0.749907  0.000975                 9
              payment_type     0.750834  0.000047                10
               order_price     0.750872  0.000010                11
  has_missing_product_info     0.751438 -0.000557                12
             freight_ratio     0.751910 -0.00102

### 2.10 H1 summary

In [35]:
# ------------------------------------------------------------
# 14. FINAL H1 SUMMARY
# ------------------------------------------------------------

# Category-specific SHAP
category_shap_all_delay_top1 = bool(
    category_summary_df[
        "delivery_delay_is_top1"
    ].all()
)

n_categories_shap_delay_top1 = int(
    category_summary_df[
        "delivery_delay_is_top1"
    ].sum()
)


# Category-specific ablation
category_ablation_all_delay_top1 = bool(
    category_delay_summary[
        "delivery_delay_is_strongest"
    ].all()
)

n_categories_ablation_delay_top1 = int(
    category_delay_summary[
        "delivery_delay_is_strongest"
    ].sum()
)


# Global SHAP rank
delay_global_shap_rank = int(
    primary_shap_importance.loc[
        primary_shap_importance["feature"]
        == "delivery_delay_days",
        "rank",
    ].iloc[0]
)


# Holdout AUC of the model selected for explanation
primary_explanation_holdout_auc = float(
    model_comparison_df.loc[
        model_comparison_df["model"]
        == primary_explanation_model_name,
        "roc_auc",
    ].iloc[0]
)


# Main H1 evidence:
# 1. global SHAP ranks delay first
# 2. global predictor ablation ranks delay first
# 3. category SHAP ranks delay first in every category
# 4. category ablation ranks delay first in every category
#
# Temporal ablation is reported as robustness rather than being
# used to define the primary H1 result.
h1_predictive_pattern_supported = bool(
    delay_global_shap_rank == 1
    and delay_largest_auc_drop
    and category_shap_all_delay_top1
    and category_ablation_all_delay_top1
)


h1_final_summary = {
    "primary_split":
        "stratified random 80/20",

    "tuning_data":
        "80% training sample only",

    "tuning_cv":
        "5-fold StratifiedKFold",

    "tuning_metric":
        "ROC-AUC",

    "lr_final_training_cv_auc":
        FINAL_LR_CV_AUC,

    "rf_final_training_cv_auc":
        FINAL_RF_CV_AUC,

    "xgb_final_training_cv_auc":
        FINAL_XGB_CV_AUC,

    "primary_explanation_model":
        primary_explanation_model_name,

    "primary_explanation_model_training_cv_auc":
        primary_explanation_cv_auc,

    "primary_explanation_model_holdout_auc":
        primary_explanation_holdout_auc,

    "global_top_shap_feature":
        primary_shap_importance.iloc[0]["feature"],

    "global_top_shap_mean_abs":
        float(
            primary_shap_importance.iloc[0]["mean_abs_shap"]
        ),

    "delivery_delay_global_shap_rank":
        delay_global_shap_rank,

    "delivery_delay_global_ablation_rank":
        delay_ablation_rank,

    "delivery_delay_global_auc_drop":
        delay_ablation_drop,

    "delivery_delay_largest_global_auc_drop":
        delay_largest_auc_drop,

    "categories_tested":
        len(category_summary_df),

    "categories_delay_top1_shap":
        n_categories_shap_delay_top1,

    "delay_top1_shap_all_categories":
        category_shap_all_delay_top1,

    "categories_delay_top1_ablation":
        n_categories_ablation_delay_top1,

    "delay_top1_ablation_all_categories":
        category_ablation_all_delay_top1,

    "temporal_delivery_delay_rank":
        temporal_delay_rank,

    "temporal_delivery_delay_auc_drop":
        temporal_delay_auc_drop,

    "temporal_delivery_delay_is_strongest":
        temporal_delay_is_strongest,

    "h1_predictive_pattern_supported":
        h1_predictive_pattern_supported,
}


pd.DataFrame([
    h1_final_summary
]).to_csv(
    f"{OUT_DIR}/h1_final_summary.csv",
    index=False,
)


print("\n" + "=" * 72)
print("H1 COMPLETE")
print("=" * 72)

print(
    f"Global delay SHAP rank: "
    f"{delay_global_shap_rank}"
)

print(
    f"Global delay ablation rank: "
    f"{delay_ablation_rank}"
)

print(
    f"Delay top SHAP feature in "
    f"{n_categories_shap_delay_top1}/"
    f"{len(category_summary_df)} categories"
)

print(
    f"Delay top ablation predictor in "
    f"{n_categories_ablation_delay_top1}/"
    f"{len(category_delay_summary)} categories"
)

print(
    f"Temporal delay ablation rank: "
    f"{temporal_delay_rank}"
)

print(
    f"H1 predictive pattern supported: "
    f"{h1_predictive_pattern_supported}"
)


H1 COMPLETE
Global delay SHAP rank: 1
Global delay ablation rank: 1
Delay top SHAP feature in 6/6 categories
Delay top ablation predictor in 6/6 categories
Temporal delay ablation rank: 1
H1 predictive pattern supported: True


---
## Part 3 — Phase 2: Asymmetry test (H2a)

*Code withheld.* The results of this part are in `output/` (files starting with `h2_`). The README explains why this part is not published.


---
## Part 4 — Product-category moderation (H2b)

*Code withheld.* The results of this part are in `output/` (files starting with `h2b_`). The README explains why this part is not published.


---
## Part 5 — Phase 3: Text analysis (H3)

*Code withheld.* The results of this part are in `output/` (files starting with `h3_`). The README explains why this part is not published.
